# ⚠️ LEGACY / NON-THESIS EVALUATION

This notebook preserves a historical or diagnostic protocol and **must not be used for cross-model thesis comparisons**. Use the authoritative notebook in `notebooks/Transformer-JEPA/othello_gpt_jepa_smoke_evaluation.ipynb` or the appropriate `notebooks/12x12/*_evaluation.ipynb` frontend for `unified_eval_v1`.

# Othello JEPA Evaluation

This Colab notebook evaluates a frozen JEPA encoder with a trained linear next-token head, then runs absolute and relative board-state probes on the JEPA encoder residual stream.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configure Project, Data, And Runs

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

PROJECT_ROOT = Path('/content/drive/Othercomputers/MyLaptop/Master_Thesis_Code')
ARTIFACT_ROOT = Path('/content/drive/MyDrive/Master_Thesis_Artifacts')
DATA_ROOT = ARTIFACT_ROOT / 'data'
RUNS_DIR = ARTIFACT_ROOT / 'runs'
CONFIGS_DIR = PROJECT_ROOT / 'configs'

assert PROJECT_ROOT.exists(), PROJECT_ROOT
assert (PROJECT_ROOT / 'pyproject.toml').exists(), PROJECT_ROOT
assert (PROJECT_ROOT / 'othello_research').is_dir(), PROJECT_ROOT
assert ARTIFACT_ROOT.exists(), ARTIFACT_ROOT
assert DATA_ROOT.exists(), DATA_ROOT
assert RUNS_DIR.exists(), RUNS_DIR
assert CONFIGS_DIR.exists(), CONFIGS_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(PROJECT_ROOT)], check=True)
os.chdir(PROJECT_ROOT)

import yaml

print('PROJECT_ROOT :', PROJECT_ROOT)
print('ARTIFACT_ROOT:', ARTIFACT_ROOT)
print('DATA_ROOT    :', DATA_ROOT)
print('RUNS_DIR     :', RUNS_DIR)
print('CONFIGS_DIR  :', CONFIGS_DIR)

# === SINGLE SOURCE OF TRUTH ===
# Change this filename to evaluate a different JEPA experiment.
CONFIG_NAME = 'jepa_v6_hard_disjoint_action.yml'
CONFIG_PATH = CONFIGS_DIR / CONFIG_NAME
assert CONFIG_PATH.exists(), CONFIG_PATH

with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    EXPERIMENT_CONFIG = yaml.safe_load(f)

BOARD_SIZE = int(EXPERIMENT_CONFIG['data']['board_size'])
JEPA_RUN_NAME = EXPERIMENT_CONFIG['experiment']['name']
N_TRAIN_CHUNKS = int(EXPERIMENT_CONFIG['data'].get('li_split_train_chunks', 200))

# === EVAL CHUNK BUDGET (override here) ===
# How many chunks to use for training the linear head and probes.
# Default = N_TRAIN_CHUNKS (preserves old behaviour). Set to a smaller int
# (e.g. 50) for fast eval ablations. Must be <= N_TRAIN_CHUNKS so the eval
# never trains the head on chunks the pretrain model never saw.
EVAL_HEAD_TRAIN_CHUNKS = N_TRAIN_CHUNKS
# EVAL_HEAD_TRAIN_CHUNKS = 50  # uncomment to override
assert EVAL_HEAD_TRAIN_CHUNKS <= N_TRAIN_CHUNKS, (
    f'EVAL_HEAD_TRAIN_CHUNKS ({EVAL_HEAD_TRAIN_CHUNKS}) must be <= '
    f'N_TRAIN_CHUNKS ({N_TRAIN_CHUNKS}).'
)

DATA_DIR = DATA_ROOT / f'othello_{BOARD_SIZE}x{BOARD_SIZE}'
JEPA_OUT_DIR = RUNS_DIR / JEPA_RUN_NAME
HEAD_OUT_DIR = JEPA_OUT_DIR / 'linear_head'
HEAD_OUT_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_HEAD_DIR = Path('/content/jepa_eval') / JEPA_RUN_NAME
LOCAL_HEAD_DIR.mkdir(parents=True, exist_ok=True)

print('CONFIG_NAME   :', CONFIG_NAME)
print('CONFIG_PATH   :', CONFIG_PATH)
print('BOARD_SIZE    :', BOARD_SIZE)
print('JEPA_RUN_NAME :', JEPA_RUN_NAME)
print('DATA_DIR      :', DATA_DIR)
print('JEPA_OUT_DIR  :', JEPA_OUT_DIR)
print('HEAD_OUT_DIR  :', HEAD_OUT_DIR)
print('LOCAL_HEAD_DIR:', LOCAL_HEAD_DIR)

assert DATA_DIR.exists(), DATA_DIR
assert JEPA_OUT_DIR.exists(), JEPA_OUT_DIR
print('EVAL_HEAD_TRAIN_CHUNKS:', EVAL_HEAD_TRAIN_CHUNKS)
print('JEPA evaluation setup OK.')


## 3. Pick which JEPA checkpoint to evaluate

In [ ]:
checkpoint_files = sorted(JEPA_OUT_DIR.glob('*.pt'))
print('Checkpoint dir:', JEPA_OUT_DIR)
print('Available JEPA checkpoint files:')
for path in checkpoint_files:
    print(' ', path.name)

CHECKPOINT_NAME = 'final.pt'
DRIVE_CKPT = JEPA_OUT_DIR / CHECKPOINT_NAME
if not DRIVE_CKPT.exists():
    assert checkpoint_files, f'No JEPA checkpoint files found in {JEPA_OUT_DIR}'
    DRIVE_CKPT = checkpoint_files[-1]
print('Selected:', DRIVE_CKPT)

## 4. Imports

In [ ]:
import json
import math
import shutil

import torch
import matplotlib.pyplot as plt

from othello_research.datasets.dataset import load_chunk
from othello_research.datasets.move_mapping import build_mappings
from othello_research.evaluation.legal_moves import (
    evaluate_legal_moves,
    replay_game_legality,
)
from othello_research.evaluation.linear_head import (
    JEPALinearHead,
    LinearHeadTrainConfig,
    evaluate_next_move_head,
    load_linear_head,
    train_linear_head,
)
from othello_research.probes.board_state import train_board_state_probes
import othello_research.training.train_jepa as tjm

def jsonable(obj):
    if isinstance(obj, dict):
        return {str(k): jsonable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [jsonable(v) for v in obj]
    if isinstance(obj, float) and math.isnan(obj):
        return None
    return obj

print('imports OK')

## 5. Restore checkpoint from Drive and load JEPA model

In [ ]:
LOCAL_CKPT = LOCAL_HEAD_DIR / f'jepa_{DRIVE_CKPT.stem}.pt'

if not LOCAL_CKPT.exists():
    print(f'Restoring {DRIVE_CKPT.stat().st_size / 1e6:.1f} MB from Drive...')
    shutil.copy2(DRIVE_CKPT, LOCAL_CKPT)
    print('done')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt = torch.load(LOCAL_CKPT, map_location=device)
jepa_model, jepa_cfg = tjm.build_model_from_checkpoint(ckpt)
jepa_model.load_state_dict(ckpt['model'])
jepa_model.to(device)
jepa_model.eval()

print('device        :', device)
print(f'Checkpoint    : {DRIVE_CKPT.name}')
print(f'Step          : {ckpt.get("step")}')
print(f'Games seen    : {ckpt.get("games_seen", 0):,}')
objective_name = ckpt.get("model_config", {}).get(
    "objective_class",
    ckpt.get("train_config", {}).get("objective_class", "jepa"),
)
predictor_name = getattr(jepa_cfg, "predictor_type", type(jepa_model.predictor).__name__)
print(f'Objective     : {objective_name}')
print(f'Predictor     : {predictor_name} ({getattr(jepa_cfg, "predictor_n_layers", "n/a")}L/{getattr(jepa_cfg, "predictor_n_heads", "n/a")}H)')
if hasattr(jepa_cfg, "prediction_horizon"):
    print(f'Horizon / EMA : K={jepa_cfg.prediction_horizon}, momentum={jepa_cfg.ema_momentum}')
elif hasattr(jepa_cfg, "num_modes"):
    print(f'Modes / EMA   : M={jepa_cfg.num_modes}, momentum={jepa_cfg.ema_momentum}')
else:
    print(f'EMA momentum  : {getattr(jepa_cfg, "ema_momentum", "n/a")}')
print(f'Context params: {sum(p.numel() for p in jepa_model.context_encoder.parameters()):,}')
print(f'Target params : {sum(p.numel() for p in jepa_model.target_encoder.parameters()):,}')

print(f'Predictor params: {sum(p.numel() for p in jepa_model.predictor.parameters()):,}')

eval_metadata = {
    'config_name': CONFIG_NAME,
    'config_path': str(CONFIG_PATH),
    'run_name': JEPA_RUN_NAME,
    'run_dir': str(JEPA_OUT_DIR),
    'checkpoint_name': DRIVE_CKPT.name,
    'checkpoint_path': str(DRIVE_CKPT),
    'checkpoint_stem': DRIVE_CKPT.stem,
    'step': ckpt.get('step'),
    'games_seen': ckpt.get('games_seen', 0),
    'tokens_seen': ckpt.get('tokens_seen', 0),
    'experiment_config': EXPERIMENT_CONFIG,
    'checkpoint_jepa_config': ckpt.get('jepa_config'),
    'checkpoint_train_config': ckpt.get('train_config'),
    'expected_outputs': {
        'next_move': str(HEAD_OUT_DIR / f'eval_next_move__{DRIVE_CKPT.stem}.json'),
        'legal_moves': str(JEPA_OUT_DIR / f'eval_legal_moves__{DRIVE_CKPT.stem}.json'),
        'board_state_probes': str(JEPA_OUT_DIR / f'eval_board_state_probes__{DRIVE_CKPT.stem}.json'),
        'plots_dir': str(JEPA_OUT_DIR / 'eval_plots'),
    },
}
metadata_path = JEPA_OUT_DIR / f'eval_metadata__{DRIVE_CKPT.stem}.json'
metadata_path.write_text(json.dumps(jsonable(eval_metadata), indent=2), encoding='utf-8')
print('metadata saved:', metadata_path)


## 6. Sanity: embedding diversity check

In [ ]:
all_chunks = sorted(DATA_DIR.glob('*.pickle'))
assert len(all_chunks) > N_TRAIN_CHUNKS, f'Need validation chunks in {DATA_DIR}'
val_chunks = all_chunks[N_TRAIN_CHUNKS:]
games = [g for g in load_chunk(val_chunks[0]) if len(g) >= 10][:16]
assert len(games) == 16, 'Need 16 validation games of length at least 10'
raw_to_token, token_to_raw = build_mappings(BOARD_SIZE)

tokens = torch.tensor([
    [raw_to_token[m] for m in g[:10]] for g in games
], device=device)

with torch.inference_mode():
    h_all = jepa_model.encode_hidden(jepa_model.context_encoder, tokens)
    h = h_all[:, -1, :]

print(f'Mean per-dim std: {h.std(dim=0).mean().item():.4f}')
h_norm = h / h.norm(dim=1, keepdim=True)
sim = h_norm @ h_norm.T
print(f'Mean off-diagonal cosine sim: {(sim.sum() - sim.diag().sum()) / (16 * 15):.4f}')
print('Healthy: std > 0.5, cosine sim < 0.5')

## 7. Sanity check: dataset next-token targets

In [ ]:
all_chunks = sorted(DATA_DIR.glob('*.pickle'))
train_chunks = all_chunks[:EVAL_HEAD_TRAIN_CHUNKS]  # head/probe training
val_chunks = all_chunks[N_TRAIN_CHUNKS:]  # val uses the pretrain split (unchanged)
print(f'Train chunks: {len(train_chunks)}, val chunks: {len(val_chunks)}')

from othello_research.datasets.dataset import TARGET_PAD
from othello_research.evaluation.linear_head import _make_loader

sanity_cfg = LinearHeadTrainConfig(
    train_chunks=train_chunks[:1],
    out_dir=str(LOCAL_HEAD_DIR),
    board_size=BOARD_SIZE,
    batch_size=4,
    n_chunks=1,
)
sanity_model = JEPALinearHead(jepa_model).to(device)
loader, games, dataset = _make_loader(train_chunks[0], sanity_model, sanity_cfg, shuffle=False)
x, y = next(iter(loader))
print('x shape:', tuple(x.shape), 'y shape:', tuple(y.shape))
print('x[0, :10] =', x[0, :10].tolist())
print('y[0, :10] =', y[0, :10].tolist())
valid_shift = y[:, :-1] != TARGET_PAD
assert torch.equal(x[:, 1:][valid_shift], y[:, :-1][valid_shift]), 'Dataset y is not x shifted by one token'
assert (y == TARGET_PAD).any() or x.shape[1] == sanity_model.config.block_size
print('Dataset next-token target check OK')

## 8. Train or load linear head on all valid next-move positions

In [ ]:
RETRAIN_LINEAR_HEAD = True
HEAD_CKPT_DRIVE = HEAD_OUT_DIR / 'linear_head.pt'
HEAD_CKPT_LOCAL = LOCAL_HEAD_DIR / 'linear_head.pt'

if HEAD_CKPT_DRIVE.exists() and not RETRAIN_LINEAR_HEAD:
    shutil.copy2(HEAD_CKPT_DRIVE, HEAD_CKPT_LOCAL)
    head_model = load_linear_head(jepa_model, HEAD_CKPT_LOCAL, device)
    print('Loaded existing linear head:', HEAD_CKPT_DRIVE)
else:
    cfg = LinearHeadTrainConfig(
        train_chunks=train_chunks,
        out_dir=str(LOCAL_HEAD_DIR),
        board_size=BOARD_SIZE,
        batch_size=512,
        learning_rate=1e-3,
        weight_decay=0.0,
        n_chunks=0,  # 0 means all training chunks
        precision='bf16',
    )
    head_model = train_linear_head(jepa_model, cfg, device)
    HEAD_OUT_DIR.mkdir(parents=True, exist_ok=True)
    for src in LOCAL_HEAD_DIR.iterdir():
        if src.is_file():
            shutil.copy2(src, HEAD_OUT_DIR / src.name)
    print('Linear head training finished and saved to:', HEAD_OUT_DIR)

assert HEAD_CKPT_LOCAL.exists(), HEAD_CKPT_LOCAL

NEXT_MOVE_EVAL_CHUNKS = 1
results_next_move = evaluate_next_move_head(
    model=head_model,
    chunks=val_chunks,
    board_size=BOARD_SIZE,
    device=device,
    n_chunks=NEXT_MOVE_EVAL_CHUNKS,
    batch_size=512,
    precision='bf16',
)
print('Validation next-move loss:', results_next_move['loss'])
print('Validation next-move top-1:', results_next_move['top1_accuracy'])
print('Validation next-move top-3:', results_next_move['top3_accuracy'])
print('Validation next-move top-5:', results_next_move['top5_accuracy'])
next_move_result_path = HEAD_OUT_DIR / f'eval_next_move__{DRIVE_CKPT.stem}.json'
next_move_result_path.write_text(json.dumps(jsonable(results_next_move), indent=2), encoding='utf-8')
print('saved:', next_move_result_path)

## 9. Legal-move evaluation using the trained head

In [ ]:
n_games = 5000
k_values = (1, 2, 3, 5)
batch_size = 64

results_legal = evaluate_legal_moves(
    model=head_model,
    val_chunks=val_chunks,
    board_size=BOARD_SIZE,
    n_games=n_games,
    device=device,
    k_values=k_values,
    batch_size=batch_size,
)

def pct(x):
    return f'{100 * x:.2f} %' if not math.isnan(float(x)) else 'nan'

print(f'Checkpoint           : {DRIVE_CKPT.name}')
print(f"Games evaluated      : {results_legal['n_games']:,}")
print(f"Tokens evaluated     : {results_legal['n_tokens']:,}")
print(f"Top-1 legal accuracy : {pct(results_legal['top1_legal_per_token'])} "
      f"(per-token) / {pct(results_legal['top1_legal_per_game'])} (per-game)")
print(f"Top-3 legal fraction : {pct(results_legal['topk_legal'][3])}")
print(f"Top-5 legal fraction : {pct(results_legal['topk_legal'][5])}")
print(f"Legal prob mass      : {pct(results_legal['legal_prob_mass_per_token'])} (per-token)")

## 10. Plots: top-1 legal, probability mass, and legal moves vs position

In [ ]:
PLOT_DIR = JEPA_OUT_DIR / 'eval_plots'
PLOT_DIR.mkdir(parents=True, exist_ok=True)

move_numbers = list(range(1, len(results_legal['top1_legal_per_position']) + 1))
plt.figure(figsize=(8, 4.5))
plt.plot(move_numbers, results_legal['top1_legal_per_position'], marker='o', markersize=3)
plt.xlabel('Move number')
plt.ylabel('Top-1 legal accuracy')
plt.ylim(0.0, 1.02)
plt.grid(True, alpha=0.3)
plt.tight_layout()
path = PLOT_DIR / f'top1_legal_accuracy__{DRIVE_CKPT.stem}.png'
plt.savefig(path, dpi=160)
plt.show()
print('saved:', path)

move_numbers = list(range(1, len(results_legal['legal_prob_mass_per_position']) + 1))
plt.figure(figsize=(8, 4.5))
plt.plot(move_numbers, results_legal['legal_prob_mass_per_position'], marker='o', markersize=3)
plt.xlabel('Move number')
plt.ylabel('Legal probability mass')
plt.ylim(0.0, 1.02)
plt.grid(True, alpha=0.3)
plt.tight_layout()
path = PLOT_DIR / f'legal_prob_mass__{DRIVE_CKPT.stem}.png'
plt.savefig(path, dpi=160)
plt.show()
print('saved:', path)

move_numbers = list(range(1, len(results_legal['n_legal_per_position_mean']) + 1))
plt.figure(figsize=(8, 4.5))
plt.plot(move_numbers, results_legal['n_legal_per_position_mean'], marker='o', markersize=3)
plt.xlabel('Move number')
plt.ylabel('Mean number of legal moves')
plt.grid(True, alpha=0.3)
plt.tight_layout()
path = PLOT_DIR / f'mean_legal_moves__{DRIVE_CKPT.stem}.png'
plt.savefig(path, dpi=160)
plt.show()
print('saved:', path)

## 11. Save legal-move results

In [ ]:
def jsonable(obj):
    if isinstance(obj, dict):
        return {str(k): jsonable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [jsonable(v) for v in obj]
    if isinstance(obj, float) and math.isnan(obj):
        return None
    return obj

legal_result_path = JEPA_OUT_DIR / f'eval_legal_moves__{DRIVE_CKPT.stem}.json'
legal_result_path.write_text(json.dumps(jsonable(results_legal), indent=2), encoding='utf-8')
print('saved:', legal_result_path)

## 12. Linear probes for ABSOLUTE and RELATIVE board state

In [ ]:
PROBE_TRAIN_GAMES = 5000
PROBE_VAL_GAMES = 512
PROBE_BATCH_SIZE = 64
PROBE_EPOCHS = 3
PROBE_LR = 1e-3

ABSOLUTE_PROBE_HIDDEN_DIM = None
RELATIVE_PROBE_HIDDEN_DIM = None

# train_board_state_probes expects the OthelloGPT interface, so pass the frozen context encoder.
PROBE_LAYERS = tuple(range(len(jepa_model.context_encoder.blocks) + 1))

probe_bank, probe_results = train_board_state_probes(
    model=jepa_model.context_encoder,
    train_chunks=train_chunks,
    val_chunks=val_chunks,
    board_size=BOARD_SIZE,
    n_train_games=PROBE_TRAIN_GAMES,
    n_val_games=PROBE_VAL_GAMES,
    device=device,
    layers=PROBE_LAYERS,
    batch_size=PROBE_BATCH_SIZE,
    epochs=PROBE_EPOCHS,
    lr=PROBE_LR,
    absolute_hidden_dim=ABSOLUTE_PROBE_HIDDEN_DIM,
    relative_hidden_dim=RELATIVE_PROBE_HIDDEN_DIM,
)

print(f"Probe train games : {probe_results['n_train_games']:,}")
print(f"Probe val games   : {probe_results['n_val_games']:,}")
print(f"Probe epochs      : {probe_results['epochs']}")

## 13. Probe summary table

In [ ]:
metrics = probe_results['metrics']
print(f"{'layer':>5s} {'abs acc':>10s} {'abs occ':>10s} {'rel acc':>10s} {'rel occ':>10s}")
for layer in probe_results['layers']:
    m = metrics[str(layer)]
    print(
        f"{layer:5d} "
        f"{m['absolute']['accuracy']:10.2%} "
        f"{m['absolute']['occupied_accuracy']:10.2%} "
        f"{m['relative']['accuracy']:10.2%} "
        f"{m['relative']['occupied_accuracy']:10.2%}"
    )

## 14. Probe plots: accuracy vs layer

In [ ]:
layers = probe_results['layers']
abs_acc = [metrics[str(layer)]['absolute']['accuracy'] for layer in layers]
rel_acc = [metrics[str(layer)]['relative']['accuracy'] for layer in layers]
abs_occ = [metrics[str(layer)]['absolute']['occupied_accuracy'] for layer in layers]
rel_occ = [metrics[str(layer)]['relative']['occupied_accuracy'] for layer in layers]

plt.figure(figsize=(8, 4.5))
plt.plot(layers, abs_acc, marker='o', label='absolute: black/white/empty')
plt.plot(layers, rel_acc, marker='o', label='relative: mine/yours/empty')
plt.xlabel('Layer')
plt.ylabel('Board-state probe accuracy')
plt.ylim(0.0, 1.02)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
path = PLOT_DIR / f'probe_accuracy_by_layer__{DRIVE_CKPT.stem}.png'
plt.savefig(path, dpi=160)
plt.show()
print('saved:', path)

plt.figure(figsize=(8, 4.5))
plt.plot(layers, abs_occ, marker='o', label='absolute occupied squares')
plt.plot(layers, rel_occ, marker='o', label='relative occupied squares')
plt.xlabel('Layer')
plt.ylabel('Occupied-square probe accuracy')
plt.ylim(0.0, 1.02)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
path = PLOT_DIR / f'probe_occupied_accuracy_by_layer__{DRIVE_CKPT.stem}.png'
plt.savefig(path, dpi=160)
plt.show()
print('saved:', path)

## 15. Save probe results

In [ ]:
if 'jsonable' not in globals():
    def jsonable(obj):
        if isinstance(obj, dict):
            return {str(k): jsonable(v) for k, v in obj.items()}
        if isinstance(obj, list):
            return [jsonable(v) for v in obj]
        if isinstance(obj, float) and math.isnan(obj):
            return None
        return obj

probe_result_path = JEPA_OUT_DIR / f'eval_board_state_probes__{DRIVE_CKPT.stem}.json'
probe_result_path.write_text(json.dumps(jsonable(probe_results), indent=2), encoding='utf-8')
print('saved:', probe_result_path)

## A. Generic MLP Probe Module

These cells add nonlinear readouts on top of the frozen JEPA context encoder. They do not modify or fine-tune the JEPA checkpoint. All encoder activations are extracted under `torch.no_grad()`, and every new artifact is saved with the checkpoint stem in its filename so existing linear results are not overwritten.


In [ ]:
import gc
import random
import time
from pathlib import Path
from collections import defaultdict
from contextlib import nullcontext

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

class MLPProbe(nn.Module):
    """Small MLP head for probing frozen representations.

    Architecture:
        n_layers=1: Linear(d_in, hidden) -> GELU -> Dropout -> Linear(hidden, d_out)
        n_layers=2: same + one extra (Linear(hidden, hidden) -> GELU -> Dropout) block

    Not a Transformer. Not attention. Pure feedforward.
    """
    def __init__(
        self,
        d_in: int,
        d_out: int,
        hidden_dim: int = 512,
        n_layers: int = 2,
        dropout: float = 0.1,
    ):
        super().__init__()
        if n_layers < 1:
            raise ValueError("n_layers must be >= 1")
        layers = [nn.Linear(d_in, hidden_dim), nn.GELU(), nn.Dropout(dropout)]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout)]
        layers += [nn.Linear(hidden_dim, d_out)]
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

MLP_PROBE_HIDDEN_DIM = 512
MLP_PROBE_N_LAYERS = 1
MLP_PROBE_DROPOUT = 0.1

def _unique_artifact_path(path: Path) -> Path:
    """Return path, or a numbered sibling if path already exists."""
    path = Path(path)
    if not path.exists():
        return path
    for idx in range(2, 1000):
        candidate = path.with_name(f'{path.stem}__rerun{idx}{path.suffix}')
        if not candidate.exists():
            return candidate
    raise RuntimeError(f'Could not find a free artifact filename for {path}')

jepa_model.eval()
for parameter in jepa_model.parameters():
    parameter.requires_grad_(False)

print('Frozen JEPA parameters:', sum(p.numel() for p in jepa_model.parameters()))
print('MLP probe config:', {
    'hidden_dim': MLP_PROBE_HIDDEN_DIM,
    'n_layers': MLP_PROBE_N_LAYERS,
    'dropout': MLP_PROBE_DROPOUT,
})


## B. MLP next-move head

This mirrors the existing linear next-move head, but replaces the single linear readout with `MLPProbe`. The target remains the sampled next move, so the task is still stochastic-bounded; train and validation metrics are reported to make overfitting visible.


In [ ]:
from othello_research.datasets.dataset import TARGET_PAD
from othello_research.evaluation.linear_head import _make_loader


# Make this new section runnable even if the earlier dataset-sanity cell was skipped.
# The split matches the existing linear evaluation convention exactly.
if 'train_chunks' not in globals() or 'val_chunks' not in globals():
    all_chunks = sorted(DATA_DIR.glob('*.pickle'))
    assert len(all_chunks) > N_TRAIN_CHUNKS, f'Need validation chunks in {DATA_DIR}'
    train_chunks = all_chunks[:N_TRAIN_CHUNKS]
    val_chunks = all_chunks[N_TRAIN_CHUNKS:]
    print(f'Initialized train/val chunks: {len(train_chunks)} train, {len(val_chunks)} val')

if 'NEXT_MOVE_EVAL_CHUNKS' not in globals():
    NEXT_MOVE_EVAL_CHUNKS = 1

class JEPAMLPNextMoveHead(nn.Module):
    """Frozen JEPA context encoder plus trainable MLP next-token head."""

    def __init__(
        self,
        jepa_model: nn.Module,
        *,
        hidden_dim: int = 512,
        n_layers: int = 1,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        self.encoder = jepa_model.context_encoder
        self.encoder_config = jepa_model.config
        for parameter in self.encoder.parameters():
            parameter.requires_grad_(False)
        self.encoder.eval()

        self.head = MLPProbe(
            self.encoder_config.d_model,
            self.encoder_config.vocab_size,
            hidden_dim=hidden_dim,
            n_layers=n_layers,
            dropout=dropout,
        )

    @property
    def config(self):
        return self.encoder_config

    def train(self, mode: bool = True) -> "JEPAMLPNextMoveHead":
        super().train(mode)
        self.encoder.eval()
        return self

    def forward(self, idx: torch.Tensor, targets: torch.Tensor | None = None):
        if idx.ndim != 2:
            raise ValueError(f"Expected idx with shape (B, T), got {tuple(idx.shape)}")
        _, seq_len = idx.shape
        if seq_len > self.config.block_size:
            raise ValueError(f"Input length {seq_len} exceeds block_size {self.config.block_size}.")

        with torch.no_grad():
            enc = self.encoder
            pos = torch.arange(0, seq_len, device=idx.device)
            x = enc.wte(idx) + enc.wpe(pos)
            x = enc.drop(x)
            for block in enc.blocks:
                x = block(x)
            x = enc.ln_f(x)

        logits = self.head(x)
        if targets is None:
            return logits, None
        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            targets.reshape(-1),
            ignore_index=TARGET_PAD,
        )
        return logits, loss


def _amp_eval_context(device: torch.device, precision: str = 'bf16'):
    if device.type == 'cuda' and precision in {'bf16', 'fp16'}:
        dtype = torch.bfloat16 if precision == 'bf16' else torch.float16
        return torch.autocast(device_type='cuda', dtype=dtype, enabled=True)
    return nullcontext()


def _next_move_batch_metrics(logits: torch.Tensor, targets: torch.Tensor, loss: torch.Tensor) -> dict[str, float]:
    mask = targets != TARGET_PAD
    n_tokens = int(mask.sum().item())
    if n_tokens == 0:
        return {'loss_sum': 0.0, 'tokens': 0, 'top1': 0, 'top3': 0, 'top5': 0}
    valid_logits = logits[mask]
    valid_targets = targets[mask]
    preds = valid_logits.argmax(dim=-1)
    k3 = min(3, logits.size(-1))
    k5 = min(5, logits.size(-1))
    top3 = torch.topk(valid_logits, k=k3, dim=-1).indices
    top5 = torch.topk(valid_logits, k=k5, dim=-1).indices
    return {
        'loss_sum': float(loss.detach().cpu().item()) * n_tokens,
        'tokens': n_tokens,
        'top1': int((preds == valid_targets).sum().item()),
        'top3': int((top3 == valid_targets.unsqueeze(-1)).any(dim=-1).sum().item()),
        'top5': int((top5 == valid_targets.unsqueeze(-1)).any(dim=-1).sum().item()),
    }


def _next_move_totals_to_metrics(totals: dict[str, float]) -> dict[str, float | int]:
    n = max(1, int(totals.get('tokens', 0)))
    return {
        'loss': float(totals.get('loss_sum', 0.0) / n),
        'top1_accuracy': float(totals.get('top1', 0) / n),
        'top3_accuracy': float(totals.get('top3', 0) / n),
        'top5_accuracy': float(totals.get('top5', 0) / n),
        'accuracy': float(totals.get('top1', 0) / n),
        'n_tokens': int(totals.get('tokens', 0)),
    }


def train_mlp_next_move_head(
    jepa_model: nn.Module,
    train_chunks,
    val_chunks,
    *,
    board_size: int,
    device: torch.device,
    batch_size: int = 512,
    lr: float = 1e-3,
    weight_decay: float = 0.0,
    epochs: int = 1,
    eval_chunks: int = 1,
    precision: str = 'bf16',
    hidden_dim: int = 512,
    n_layers: int = 2,
    dropout: float = 0.1,
):
    model = JEPAMLPNextMoveHead(
        jepa_model,
        hidden_dim=hidden_dim,
        n_layers=n_layers,
        dropout=dropout,
    ).to(device)
    optimizer = torch.optim.AdamW(model.head.parameters(), lr=lr, weight_decay=weight_decay)
    history = []

    train_cfg = LinearHeadTrainConfig(
        train_chunks=train_chunks,
        out_dir=str(LOCAL_HEAD_DIR),
        board_size=board_size,
        batch_size=batch_size,
        learning_rate=lr,
        weight_decay=weight_decay,
        n_chunks=0,
        precision=precision,
    )

    for epoch in range(1, epochs + 1):
        model.train()
        totals = defaultdict(float)
        pbar = tqdm(train_chunks, desc=f'mlp next-move epoch {epoch}/{epochs}', unit='chunk')
        for chunk_path in pbar:
            loader, games, dataset = _make_loader(chunk_path, model, train_cfg, shuffle=True)
            for x_cpu, y_cpu in tqdm(loader, desc=f'mlp {Path(chunk_path).name}', leave=False):
                x = x_cpu.to(device, non_blocking=device.type == 'cuda')
                y = y_cpu.to(device, non_blocking=device.type == 'cuda')
                optimizer.zero_grad(set_to_none=True)
                with _amp_eval_context(device, precision):
                    logits, loss = model(x, y)
                if loss is None:
                    raise RuntimeError('MLP next-move training expected a loss')
                loss.backward()
                optimizer.step()

                metrics = _next_move_batch_metrics(logits, y, loss)
                for key, value in metrics.items():
                    totals[key] += value
            del loader, dataset, games
            gc.collect()
            if device.type == 'cuda':
                torch.cuda.empty_cache()
            train_running = _next_move_totals_to_metrics(totals)
            pbar.set_postfix(loss=f"{train_running['loss']:.4f}", top1=f"{train_running['top1_accuracy']:.2%}")

        train_metrics = _next_move_totals_to_metrics(totals)
        val_metrics = evaluate_next_move_head(
            model=model,
            chunks=val_chunks,
            board_size=board_size,
            device=device,
            n_chunks=eval_chunks,
            batch_size=batch_size,
            precision=precision,
        )
        row = {'epoch': epoch, 'train': train_metrics, 'val': val_metrics}
        history.append(row)
        print(
            f"epoch={epoch} train_loss={train_metrics['loss']:.4f} "
            f"train_top1={train_metrics['top1_accuracy']:.2%} "
            f"val_loss={val_metrics['loss']:.4f} val_top1={val_metrics['top1_accuracy']:.2%}"
        )

    return model, history

RETRAIN_MLP_NEXT_MOVE_HEAD = True
MLP_NEXT_MOVE_EPOCHS = 1
MLP_NEXT_MOVE_BATCH_SIZE = 512
MLP_NEXT_MOVE_LR = 1e-3
MLP_NEXT_MOVE_WEIGHT_DECAY = 0.0
MLP_NEXT_MOVE_PRECISION = 'bf16'

MLP_NEXT_MOVE_HEAD_BASE = HEAD_OUT_DIR / f'mlp_next_move_head__{DRIVE_CKPT.stem}.pt'
MLP_NEXT_MOVE_JSON_BASE = HEAD_OUT_DIR / f'eval_next_move_mlp__{DRIVE_CKPT.stem}.json'
MLP_NEXT_MOVE_HEAD_DRIVE = _unique_artifact_path(MLP_NEXT_MOVE_HEAD_BASE) if RETRAIN_MLP_NEXT_MOVE_HEAD else MLP_NEXT_MOVE_HEAD_BASE
MLP_NEXT_MOVE_JSON = _unique_artifact_path(MLP_NEXT_MOVE_JSON_BASE) if RETRAIN_MLP_NEXT_MOVE_HEAD else MLP_NEXT_MOVE_JSON_BASE

if MLP_NEXT_MOVE_HEAD_DRIVE.exists() and not RETRAIN_MLP_NEXT_MOVE_HEAD:
    mlp_next_move_model = JEPAMLPNextMoveHead(
        jepa_model,
        hidden_dim=MLP_PROBE_HIDDEN_DIM,
        n_layers=MLP_PROBE_N_LAYERS,
        dropout=MLP_PROBE_DROPOUT,
    ).to(device)
    payload = torch.load(MLP_NEXT_MOVE_HEAD_DRIVE, map_location=device)
    mlp_next_move_model.head.load_state_dict(payload['head'])
    mlp_next_move_model.eval()
    mlp_next_move_history = payload.get('history', [])
    print('Loaded existing MLP next-move head:', MLP_NEXT_MOVE_HEAD_DRIVE)
else:
    mlp_next_move_model, mlp_next_move_history = train_mlp_next_move_head(
        jepa_model,
        train_chunks,
        val_chunks,
        board_size=BOARD_SIZE,
        device=device,
        batch_size=MLP_NEXT_MOVE_BATCH_SIZE,
        lr=MLP_NEXT_MOVE_LR,
        weight_decay=MLP_NEXT_MOVE_WEIGHT_DECAY,
        epochs=MLP_NEXT_MOVE_EPOCHS,
        eval_chunks=NEXT_MOVE_EVAL_CHUNKS,
        precision=MLP_NEXT_MOVE_PRECISION,
        hidden_dim=MLP_PROBE_HIDDEN_DIM,
        n_layers=MLP_PROBE_N_LAYERS,
        dropout=MLP_PROBE_DROPOUT,
    )
    torch.save(
        {
            'head': mlp_next_move_model.head.state_dict(),
            'head_type': 'mlp_next_move',
            'checkpoint_path': str(DRIVE_CKPT),
            'hidden_dim': MLP_PROBE_HIDDEN_DIM,
            'n_layers': MLP_PROBE_N_LAYERS,
            'dropout': MLP_PROBE_DROPOUT,
            'history': mlp_next_move_history,
        },
        MLP_NEXT_MOVE_HEAD_DRIVE,
    )
    print('saved:', MLP_NEXT_MOVE_HEAD_DRIVE)

mlp_next_move_val = evaluate_next_move_head(
    model=mlp_next_move_model,
    chunks=val_chunks,
    board_size=BOARD_SIZE,
    device=device,
    n_chunks=NEXT_MOVE_EVAL_CHUNKS,
    batch_size=MLP_NEXT_MOVE_BATCH_SIZE,
    precision=MLP_NEXT_MOVE_PRECISION,
)
mlp_next_move_results = {
    'checkpoint_path': str(DRIVE_CKPT),
    'head_type': 'mlp_next_move',
    'hidden_dim': MLP_PROBE_HIDDEN_DIM,
    'n_layers': MLP_PROBE_N_LAYERS,
    'dropout': MLP_PROBE_DROPOUT,
    'train_chunks': [str(p) for p in train_chunks],
    'val_chunks': [str(p) for p in val_chunks[:NEXT_MOVE_EVAL_CHUNKS]],
    'layer_evaluated': 'all token positions from final encoder layer',
    'history': mlp_next_move_history,
    'final_train': mlp_next_move_history[-1]['train'] if mlp_next_move_history else None,
    'final_val': mlp_next_move_val,
}
MLP_NEXT_MOVE_JSON.write_text(json.dumps(jsonable(mlp_next_move_results), indent=2), encoding='utf-8')
print('Validation MLP next-move loss:', mlp_next_move_val['loss'])
print('Validation MLP next-move top-1:', mlp_next_move_val['top1_accuracy'])
print('Validation MLP next-move top-3:', mlp_next_move_val['top3_accuracy'])
print('Validation MLP next-move top-5:', mlp_next_move_val['top5_accuracy'])
print('saved:', MLP_NEXT_MOVE_JSON)


## C. Shared frozen-activation probe dataset

The next cells build one shared set of sampled `(game, position)` examples for the new MLP board probe and the legal-mask probes. This keeps the splits identical to the existing linear evaluations (`train_chunks` and `val_chunks`) while avoiding repeated game replay. By default we sample one valid post-move position per game for the new probes; increase `NEW_PROBE_POSITIONS_PER_GAME` if you want denser coverage.


In [ ]:
from othello_research.othello.board import OthelloBoardState
from othello_research.probes.board_state import (
    ActivationCache,
    EMPTY as BOARD_EMPTY_LABEL,
    board_state_labels_after_moves,
    load_games_from_chunks,
)


def _legal_moves_after_prefix(prefix_raw, board_size: int) -> list[int]:
    board = OthelloBoardState(n=board_size)
    for move in prefix_raw:
        board.umpire(int(move))
    legal = board.get_valid_moves()
    if legal:
        return list(legal)
    board.next_hand_color *= -1
    legal = list(board.get_valid_moves())
    board.next_hand_color *= -1
    return legal


def _sample_probe_examples(
    chunks,
    *,
    n_games: int,
    board_size: int,
    block_size: int,
    positions_per_game: int,
    seed: int,
):
    raw_to_token, _ = build_mappings(board_size)
    n_squares = board_size ** 2
    games = load_games_from_chunks(chunks, n_games)
    rng = random.Random(seed)
    examples = []
    skipped = 0

    for game_idx, game_raw in enumerate(games):
        raw = list(game_raw[: block_size + 1])
        max_moves = min(len(raw), block_size + 1)
        # Need at least one encoded move and one held-out next move.
        if max_moves < 2:
            skipped += 1
            continue
        for _ in range(positions_per_game):
            ply = rng.randrange(0, max_moves - 1)
            prefix_raw = raw[: ply + 1]
            try:
                tokens = [raw_to_token[int(move)] for move in prefix_raw]
            except Exception:
                skipped += 1
                continue
            if any(token < 0 for token in tokens):
                skipped += 1
                continue

            labels_abs, labels_rel = board_state_labels_after_moves(prefix_raw, board_size)
            legal_moves = _legal_moves_after_prefix(prefix_raw, board_size)
            if not legal_moves:
                skipped += 1
                continue
            legal = torch.zeros(n_squares, dtype=torch.float32)
            legal[torch.tensor(legal_moves, dtype=torch.long)] = 1.0
            examples.append({
                'tokens': torch.tensor(tokens, dtype=torch.long),
                'position': len(tokens) - 1,
                'absolute': torch.tensor(labels_abs[-1], dtype=torch.long),
                'relative': torch.tensor(labels_rel[-1], dtype=torch.long),
                'legal': legal,
                'game_idx': game_idx,
                'ply': ply,
            })
    return examples, {'loaded_games': len(games), 'examples': len(examples), 'skipped': skipped}


def collect_frozen_probe_dataset(
    encoder: nn.Module,
    chunks,
    *,
    n_games: int,
    board_size: int,
    layers,
    batch_size: int,
    device: torch.device,
    seed: int,
    positions_per_game: int = 1,
    precision: str = 'bf16',
):
    encoder.eval()
    for parameter in encoder.parameters():
        parameter.requires_grad_(False)

    block_size = int(getattr(encoder.config, 'block_size', board_size ** 2 - 5))
    pad_token = board_size ** 2 - 4
    n_squares = board_size ** 2
    layers = tuple(int(layer) for layer in layers)
    examples, sample_info = _sample_probe_examples(
        chunks,
        n_games=n_games,
        board_size=board_size,
        block_size=block_size,
        positions_per_game=positions_per_game,
        seed=seed,
    )
    if not examples:
        raise ValueError('No probe examples were sampled')

    acts_by_layer = {str(layer): [] for layer in layers}
    absolute_rows = []
    relative_rows = []
    legal_rows = []
    meta_rows = []

    with ActivationCache(encoder, layers) as cache:
        for start in tqdm(range(0, len(examples), batch_size), desc='collect frozen activations'):
            batch = examples[start : start + batch_size]
            max_len = max(len(ex['tokens']) for ex in batch)
            x = torch.full((len(batch), max_len), pad_token, dtype=torch.long)
            positions = torch.empty(len(batch), dtype=torch.long)
            for row, ex in enumerate(batch):
                tokens = ex['tokens']
                x[row, : len(tokens)] = tokens
                positions[row] = int(ex['position'])
                absolute_rows.append(ex['absolute'])
                relative_rows.append(ex['relative'])
                legal_rows.append(ex['legal'])
                meta_rows.append({'game_idx': int(ex['game_idx']), 'ply': int(ex['ply'])})

            cache.clear()
            x = x.to(device)
            positions_device = positions.to(device)
            with torch.no_grad():
                with _amp_eval_context(device, precision):
                    encoder(x)
            row_idx = torch.arange(len(batch), device=device)
            for layer in layers:
                layer_acts = cache.activations[layer][row_idx, positions_device].detach().float().cpu()
                acts_by_layer[str(layer)].append(layer_acts)

            del x
            if device.type == 'cuda':
                torch.cuda.empty_cache()

    dataset = {
        'activations': {layer: torch.cat(chunks, dim=0) for layer, chunks in acts_by_layer.items()},
        'absolute': torch.stack(absolute_rows, dim=0),
        'relative': torch.stack(relative_rows, dim=0),
        'legal': torch.stack(legal_rows, dim=0),
        'metadata': meta_rows,
        'sample_info': sample_info,
        'board_size': board_size,
        'n_squares': n_squares,
        'layers': list(layers),
        'positions_per_game': positions_per_game,
    }
    return dataset


# Make this new section runnable even if the earlier linear board-probe config cell was skipped.
# Defaults match the existing linear probe cell in this notebook.
if 'PROBE_TRAIN_GAMES' not in globals():
    PROBE_TRAIN_GAMES = 5000
if 'PROBE_VAL_GAMES' not in globals():
    PROBE_VAL_GAMES = 512
if 'PROBE_BATCH_SIZE' not in globals():
    PROBE_BATCH_SIZE = 64
if 'PROBE_EPOCHS' not in globals():
    PROBE_EPOCHS = 3
if 'PROBE_LR' not in globals():
    PROBE_LR = 1e-3
if 'PROBE_LAYERS' not in globals():
    PROBE_LAYERS = tuple(range(len(jepa_model.context_encoder.blocks) + 1))
if 'PLOT_DIR' not in globals():
    PLOT_DIR = JEPA_OUT_DIR / 'eval_plots'
    PLOT_DIR.mkdir(parents=True, exist_ok=True)

NEW_PROBE_POSITIONS_PER_GAME = 1
NEW_PROBE_SEED = 12345
NEW_PROBE_BATCH_SIZE = PROBE_BATCH_SIZE

new_probe_train_data = collect_frozen_probe_dataset(
    jepa_model.context_encoder,
    train_chunks,
    n_games=PROBE_TRAIN_GAMES,
    board_size=BOARD_SIZE,
    layers=PROBE_LAYERS,
    batch_size=NEW_PROBE_BATCH_SIZE,
    device=device,
    seed=NEW_PROBE_SEED,
    positions_per_game=NEW_PROBE_POSITIONS_PER_GAME,
)
new_probe_val_data = collect_frozen_probe_dataset(
    jepa_model.context_encoder,
    val_chunks,
    n_games=PROBE_VAL_GAMES,
    board_size=BOARD_SIZE,
    layers=PROBE_LAYERS,
    batch_size=NEW_PROBE_BATCH_SIZE,
    device=device,
    seed=NEW_PROBE_SEED + 1,
    positions_per_game=NEW_PROBE_POSITIONS_PER_GAME,
)
print('new probe train examples:', new_probe_train_data['sample_info'])
print('new probe val examples  :', new_probe_val_data['sample_info'])
print('activation shape L0 train:', tuple(new_probe_train_data['activations'][str(PROBE_LAYERS[0])].shape))


## D. MLP board-representation probe

This trains one MLP board-state probe per layer using the same absolute and relative label conventions as the existing linear board probe. Train and validation metrics are both saved; a large gap means the nonlinear probe is memorizing frozen features rather than finding a robust board representation.


In [ ]:
class MLPBoardProbe(nn.Module):
    def __init__(self, d_model: int, n_squares: int, *, hidden_dim: int, n_layers: int, dropout: float):
        super().__init__()
        self.n_squares = n_squares
        self.proj = MLPProbe(d_model, n_squares * 3, hidden_dim=hidden_dim, n_layers=n_layers, dropout=dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        logits = self.proj(x)
        return logits.view(*logits.shape[:-1], self.n_squares, 3)


def _board_counts(logits: torch.Tensor, labels: torch.Tensor) -> dict[str, int]:
    preds = logits.argmax(dim=-1)
    valid = torch.ones_like(labels, dtype=torch.bool)
    occupied = valid & (labels != BOARD_EMPTY_LABEL)
    empty = valid & (labels == BOARD_EMPTY_LABEL)
    return {
        'total': int(valid.sum().item()),
        'correct': int(((preds == labels) & valid).sum().item()),
        'occupied_total': int(occupied.sum().item()),
        'occupied_correct': int(((preds == labels) & occupied).sum().item()),
        'empty_total': int(empty.sum().item()),
        'empty_correct': int(((preds == labels) & empty).sum().item()),
    }


def _merge_int_counts(dst, src):
    for key, value in src.items():
        dst[key] = dst.get(key, 0) + int(value)


def _counts_to_probe_metrics(counts: dict[str, int]) -> dict[str, float | int]:
    def ratio(num, den):
        den_value = counts.get(den, 0)
        return float('nan') if den_value == 0 else float(counts.get(num, 0) / den_value)
    return {
        'accuracy': ratio('correct', 'total'),
        'occupied_accuracy': ratio('occupied_correct', 'occupied_total'),
        'empty_accuracy': ratio('empty_correct', 'empty_total'),
        'n_labels': int(counts.get('total', 0)),
        'n_occupied_labels': int(counts.get('occupied_total', 0)),
        'n_empty_labels': int(counts.get('empty_total', 0)),
    }


def _evaluate_board_probe_pair(abs_probe, rel_probe, X, y_abs, y_rel, *, batch_size: int, device: torch.device):
    abs_probe.eval(); rel_probe.eval()
    abs_counts, rel_counts = {}, {}
    total_loss = 0.0
    total_batches = 0
    loader = DataLoader(TensorDataset(X, y_abs, y_rel), batch_size=batch_size, shuffle=False)
    with torch.no_grad():
        for xb, ya, yr in loader:
            xb = xb.to(device)
            ya = ya.to(device)
            yr = yr.to(device)
            abs_logits = abs_probe(xb)
            rel_logits = rel_probe(xb)
            loss = 0.5 * (
                F.cross_entropy(abs_logits.reshape(-1, 3), ya.reshape(-1)) +
                F.cross_entropy(rel_logits.reshape(-1, 3), yr.reshape(-1))
            )
            total_loss += float(loss.detach().cpu())
            total_batches += 1
            _merge_int_counts(abs_counts, _board_counts(abs_logits, ya))
            _merge_int_counts(rel_counts, _board_counts(rel_logits, yr))
    return {
        'loss': total_loss / max(1, total_batches),
        'absolute': _counts_to_probe_metrics(abs_counts),
        'relative': _counts_to_probe_metrics(rel_counts),
    }


def train_mlp_board_probes(train_data, val_data, *, layers, device, epochs, batch_size, lr, weight_decay):
    n_squares = int(train_data['n_squares'])
    results = {
        'checkpoint_path': str(DRIVE_CKPT),
        'probe_type': 'mlp_board',
        'hidden_dim': MLP_PROBE_HIDDEN_DIM,
        'n_layers': MLP_PROBE_N_LAYERS,
        'dropout': MLP_PROBE_DROPOUT,
        'layers': [int(layer) for layer in layers],
        'n_train_examples': int(train_data['absolute'].shape[0]),
        'n_val_examples': int(val_data['absolute'].shape[0]),
        'positions_per_game': int(train_data['positions_per_game']),
        'train_sample_info': train_data['sample_info'],
        'val_sample_info': val_data['sample_info'],
        'metrics': {},
        'history': {},
    }
    states = {'absolute': {}, 'relative': {}, 'metadata': {k: v for k, v in results.items() if k not in {'metrics', 'history'}}}

    for layer in tqdm(layers, desc='mlp board probes by layer'):
        key = str(layer)
        X_train = train_data['activations'][key]
        X_val = val_data['activations'][key]
        y_abs_train = train_data['absolute']
        y_rel_train = train_data['relative']
        y_abs_val = val_data['absolute']
        y_rel_val = val_data['relative']
        d_model = int(X_train.shape[-1])

        abs_probe = MLPBoardProbe(d_model, n_squares, hidden_dim=MLP_PROBE_HIDDEN_DIM, n_layers=MLP_PROBE_N_LAYERS, dropout=MLP_PROBE_DROPOUT).to(device)
        rel_probe = MLPBoardProbe(d_model, n_squares, hidden_dim=MLP_PROBE_HIDDEN_DIM, n_layers=MLP_PROBE_N_LAYERS, dropout=MLP_PROBE_DROPOUT).to(device)
        optimizer = torch.optim.AdamW(list(abs_probe.parameters()) + list(rel_probe.parameters()), lr=lr, weight_decay=weight_decay)
        loader = DataLoader(TensorDataset(X_train, y_abs_train, y_rel_train), batch_size=batch_size, shuffle=True)
        layer_history = []

        for epoch in range(1, epochs + 1):
            abs_probe.train(); rel_probe.train()
            epoch_loss = 0.0
            n_batches = 0
            for xb, ya, yr in loader:
                xb = xb.to(device)
                ya = ya.to(device)
                yr = yr.to(device)
                optimizer.zero_grad(set_to_none=True)
                abs_logits = abs_probe(xb)
                rel_logits = rel_probe(xb)
                loss = 0.5 * (
                    F.cross_entropy(abs_logits.reshape(-1, 3), ya.reshape(-1)) +
                    F.cross_entropy(rel_logits.reshape(-1, 3), yr.reshape(-1))
                )
                loss.backward()
                optimizer.step()
                epoch_loss += float(loss.detach().cpu())
                n_batches += 1
            train_metrics = _evaluate_board_probe_pair(abs_probe, rel_probe, X_train, y_abs_train, y_rel_train, batch_size=batch_size, device=device)
            val_metrics = _evaluate_board_probe_pair(abs_probe, rel_probe, X_val, y_abs_val, y_rel_val, batch_size=batch_size, device=device)
            layer_history.append({'epoch': epoch, 'train_loss': epoch_loss / max(1, n_batches), 'train': train_metrics, 'val': val_metrics})
            print(
                f"layer={layer} epoch={epoch} "
                f"train_rel={train_metrics['relative']['accuracy']:.2%} "
                f"val_rel={val_metrics['relative']['accuracy']:.2%} "
                f"train_abs={train_metrics['absolute']['accuracy']:.2%} "
                f"val_abs={val_metrics['absolute']['accuracy']:.2%}"
            )

        results['metrics'][key] = {'train': layer_history[-1]['train'], 'val': layer_history[-1]['val']}
        results['history'][key] = layer_history
        states['absolute'][key] = abs_probe.cpu().state_dict()
        states['relative'][key] = rel_probe.cpu().state_dict()
        del abs_probe, rel_probe, optimizer, loader
        gc.collect()
        if device.type == 'cuda':
            torch.cuda.empty_cache()

    return states, results

mlp_board_states, mlp_board_results = train_mlp_board_probes(
    new_probe_train_data,
    new_probe_val_data,
    layers=PROBE_LAYERS,
    device=device,
    epochs=PROBE_EPOCHS,
    batch_size=PROBE_BATCH_SIZE,
    lr=PROBE_LR,
    weight_decay=0.0,
)

mlp_board_path = _unique_artifact_path(JEPA_OUT_DIR / f'mlp_board_probe__{DRIVE_CKPT.stem}.pt')
mlp_board_json = _unique_artifact_path(JEPA_OUT_DIR / f'eval_board_mlp__{DRIVE_CKPT.stem}.json')
torch.save(mlp_board_states, mlp_board_path)
mlp_board_json.write_text(json.dumps(jsonable(mlp_board_results), indent=2), encoding='utf-8')
print('saved:', mlp_board_path)
print('saved:', mlp_board_json)


## E. Linear legal-mask probe

The legal-mask target is a binary vector over board cells. This directly tests whether the frozen representation contains legality information, avoiding the stochastic ceiling of next-move prediction.


In [ ]:
def _legal_mask_metrics_from_logits(logits: torch.Tensor, labels: torch.Tensor) -> dict[str, float | int]:
    labels = labels.float()
    probs = torch.sigmoid(logits.float())
    loss = F.binary_cross_entropy_with_logits(logits.float(), labels, reduction='mean')
    pred_binary = probs >= 0.5
    label_binary = labels >= 0.5

    tp = (pred_binary & label_binary).sum(dim=1).float()
    fp = (pred_binary & ~label_binary).sum(dim=1).float()
    fn = (~pred_binary & label_binary).sum(dim=1).float()
    f1 = (2 * tp / (2 * tp + fp + fn).clamp_min(1e-12)).mean()

    def topk_hit(k: int) -> torch.Tensor:
        k = min(k, logits.size(-1))
        top_idx = torch.topk(logits, k=k, dim=-1).indices
        hits = label_binary.gather(1, top_idx).any(dim=1).float()
        return hits.mean()

    legal_mass = ((probs * labels).sum(dim=1) / probs.sum(dim=1).clamp_min(1e-12)).mean()

    auc_values = []
    scores_cpu = probs.detach().cpu()
    labels_cpu = label_binary.detach().cpu()
    for row_scores, row_labels in zip(scores_cpu, labels_cpu):
        pos = row_scores[row_labels]
        neg = row_scores[~row_labels]
        if pos.numel() == 0 or neg.numel() == 0:
            continue
        comparisons = (pos[:, None] > neg[None, :]).float()
        ties = (pos[:, None] == neg[None, :]).float()
        auc_values.append(float((comparisons + 0.5 * ties).mean().item()))
    auroc = float(sum(auc_values) / len(auc_values)) if auc_values else float('nan')

    return {
        'bce_loss': float(loss.detach().cpu().item()),
        'f1': float(f1.detach().cpu().item()),
        'top1_legal_hit_rate': float(topk_hit(1).detach().cpu().item()),
        'top3_legal_hit_rate': float(topk_hit(3).detach().cpu().item()),
        'top5_legal_hit_rate': float(topk_hit(5).detach().cpu().item()),
        'legal_probability_mass': float(legal_mass.detach().cpu().item()),
        'auroc': auroc,
        'n_examples': int(labels.shape[0]),
    }


def _evaluate_legal_probe(probe: nn.Module, X: torch.Tensor, y: torch.Tensor, *, batch_size: int, device: torch.device):
    probe.eval()
    weighted = defaultdict(float)
    total_examples = 0
    loader = DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=False)
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = probe(xb)
            metrics = _legal_mask_metrics_from_logits(logits, yb)
            n = int(metrics['n_examples'])
            total_examples += n
            for key, value in metrics.items():
                if key != 'n_examples':
                    weighted[key] += float(value) * n
    return {key: (value / max(1, total_examples)) for key, value in weighted.items()} | {'n_examples': total_examples}


def train_legal_mask_probes(
    train_data,
    val_data,
    *,
    layers,
    device,
    epochs,
    batch_size,
    lr,
    weight_decay,
    probe_kind: str,
):
    n_squares = int(train_data['n_squares'])
    # Legal-mask MLP probes can memorize small frozen-feature probing sets.
    # Use stricter defaults for this probe family only; override these globals
    # in the notebook if you want to run a capacity/regularization ablation.
    legal_mlp_hidden_dim = int(globals().get('LEGAL_MASK_MLP_HIDDEN_DIM', 128))
    legal_mlp_dropout = float(globals().get('LEGAL_MASK_MLP_DROPOUT', 0.5))
    legal_mlp_weight_decay = float(globals().get('LEGAL_MASK_MLP_WEIGHT_DECAY', 1e-2))
    legal_mlp_lr = float(globals().get('LEGAL_MASK_MLP_LR', min(float(lr), 3e-4)))
    legal_mlp_patience = int(globals().get('LEGAL_MASK_MLP_EARLY_STOP_PATIENCE', 1))
    legal_mlp_input_noise_std = float(globals().get('LEGAL_MASK_MLP_INPUT_NOISE_STD', 0.03))
    legal_mask_grad_clip_norm = float(globals().get('LEGAL_MASK_GRAD_CLIP_NORM', 1.0))

    results = {
        'checkpoint_path': str(DRIVE_CKPT),
        'probe_type': f'{probe_kind}_legal_mask',
        'hidden_dim': legal_mlp_hidden_dim if probe_kind == 'mlp' else None,
        'n_layers': MLP_PROBE_N_LAYERS if probe_kind == 'mlp' else None,
        'dropout': legal_mlp_dropout if probe_kind == 'mlp' else None,
        'effective_lr': legal_mlp_lr if probe_kind == 'mlp' else lr,
        'effective_weight_decay': legal_mlp_weight_decay if probe_kind == 'mlp' else weight_decay,
        'input_noise_std': legal_mlp_input_noise_std if probe_kind == 'mlp' else 0.0,
        'grad_clip_norm': legal_mask_grad_clip_norm if probe_kind == 'mlp' else 0.0,
        'layers': [int(layer) for layer in layers],
        'n_train_examples': int(train_data['legal'].shape[0]),
        'n_val_examples': int(val_data['legal'].shape[0]),
        'positions_per_game': int(train_data['positions_per_game']),
        'train_sample_info': train_data['sample_info'],
        'val_sample_info': val_data['sample_info'],
        'metrics': {},
        'history': {},
    }
    states = {'layers': {}, 'metadata': {k: v for k, v in results.items() if k not in {'metrics', 'history'}}}

    for layer in tqdm(layers, desc=f'{probe_kind} legal-mask probes by layer'):
        key = str(layer)
        X_train = train_data['activations'][key]
        X_val = val_data['activations'][key]
        y_train = train_data['legal']
        y_val = val_data['legal']
        d_model = int(X_train.shape[-1])
        if probe_kind == 'linear':
            probe = nn.Linear(d_model, n_squares)
        elif probe_kind == 'mlp':
            probe = MLPProbe(
                d_model,
                n_squares,
                hidden_dim=legal_mlp_hidden_dim,
                n_layers=MLP_PROBE_N_LAYERS,
                dropout=legal_mlp_dropout,
            )
        else:
            raise ValueError(f'unknown probe_kind={probe_kind}')
        probe = probe.to(device)
        effective_lr = legal_mlp_lr if probe_kind == 'mlp' else lr
        effective_weight_decay = legal_mlp_weight_decay if probe_kind == 'mlp' else weight_decay
        effective_grad_clip_norm = legal_mask_grad_clip_norm if probe_kind == 'mlp' else 0.0
        optimizer = torch.optim.AdamW(
            probe.parameters(),
            lr=effective_lr,
            weight_decay=effective_weight_decay,
        )
        loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
        layer_history = []

        # Multi-label legal-mask targets are highly imbalanced: most cells are illegal.
        # Weight positive labels during training only; evaluation metrics remain unchanged.
        pos = y_train.float().sum(dim=0)
        neg = y_train.shape[0] - pos
        pos_weight = (neg / pos.clamp_min(1.0)).clamp(max=20.0).to(device)

        patience = legal_mlp_patience if probe_kind == 'mlp' else int(globals().get('LEGAL_MASK_EARLY_STOP_PATIENCE', 2))
        best_score = -float('inf')
        best_epoch = None
        best_state = None
        best_metrics = None
        no_improve_epochs = 0

        for epoch in range(1, epochs + 1):
            probe.train()  # keeps MLP dropout active during probe training
            epoch_loss = 0.0
            n_batches = 0
            for xb, yb in loader:
                xb = xb.to(device)
                yb = yb.to(device)
                optimizer.zero_grad(set_to_none=True)
                if probe_kind == 'mlp' and legal_mlp_input_noise_std > 0:
                    xb_in = xb + torch.randn_like(xb) * legal_mlp_input_noise_std
                else:
                    xb_in = xb
                logits = probe(xb_in)
                loss = F.binary_cross_entropy_with_logits(
                    logits.float(),
                    yb.float(),
                    pos_weight=pos_weight,
                )
                loss.backward()
                if effective_grad_clip_norm > 0:
                    torch.nn.utils.clip_grad_norm_(probe.parameters(), effective_grad_clip_norm)
                optimizer.step()
                epoch_loss += float(loss.detach().cpu())
                n_batches += 1
            train_loss = epoch_loss / max(1, n_batches)
            train_metrics = _evaluate_legal_probe(probe, X_train, y_train, batch_size=batch_size, device=device)
            val_metrics = _evaluate_legal_probe(probe, X_val, y_val, batch_size=batch_size, device=device)
            val_loss = float(val_metrics['bce_loss'])
            score = float(val_metrics['top1_legal_hit_rate'])
            improved = score > best_score
            if improved:
                best_score = score
                best_epoch = epoch
                best_state = {
                    k: v.detach().cpu().clone()
                    for k, v in probe.state_dict().items()
                }
                best_metrics = {
                    'train': train_metrics,
                    'val': val_metrics,
                }
                no_improve_epochs = 0
            else:
                no_improve_epochs += 1

            layer_history.append({
                'epoch': epoch,
                'train_loss': train_loss,
                'val_loss': val_loss,
                'score': score,
                'improved': improved,
                'train': train_metrics,
                'val': val_metrics,
            })
            print(
                f"{probe_kind} layer={layer} epoch={epoch} "
                f"train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
                f"lr={effective_lr:.1e} wd={effective_weight_decay:.1e} "
                f"train_top1={train_metrics['top1_legal_hit_rate']:.2%} "
                f"val_top1={val_metrics['top1_legal_hit_rate']:.2%} "
                f"train_f1={train_metrics['f1']:.3f} val_f1={val_metrics['f1']:.3f} "
                f"best_epoch={best_epoch}"
            )
            if no_improve_epochs >= patience:
                print(f"early stopping layer={layer} epoch={epoch}")
                break

        if best_state is None or best_metrics is None or best_epoch is None:
            raise RuntimeError(f'No best checkpoint was selected for layer={layer}')
        results['metrics'][key] = {
            'best_epoch': int(best_epoch),
            'best_score': float(best_score),
            'train': best_metrics['train'],
            'val': best_metrics['val'],
        }
        results['history'][key] = layer_history
        states['layers'][key] = best_state
        del probe, optimizer, loader
        gc.collect()
        if device.type == 'cuda':
            torch.cuda.empty_cache()

    return states, results

linear_legal_states, linear_legal_results = train_legal_mask_probes(
    new_probe_train_data,
    new_probe_val_data,
    layers=PROBE_LAYERS,
    device=device,
    epochs=PROBE_EPOCHS,
    batch_size=PROBE_BATCH_SIZE,
    lr=PROBE_LR,
    weight_decay=0.0,
    probe_kind='linear',
)

linear_legal_path = _unique_artifact_path(JEPA_OUT_DIR / f'linear_legal_mask_probe__{DRIVE_CKPT.stem}.pt')
linear_legal_json = _unique_artifact_path(JEPA_OUT_DIR / f'eval_legal_mask_linear__{DRIVE_CKPT.stem}.json')
torch.save(linear_legal_states, linear_legal_path)
linear_legal_json.write_text(json.dumps(jsonable(linear_legal_results), indent=2), encoding='utf-8')
print('saved:', linear_legal_path)
print('saved:', linear_legal_json)


## F. MLP legal-mask probe

This uses the same legal-mask targets and sampled positions as the linear legal-mask probe, but swaps the readout to `MLPProbe`. Compare train and validation metrics against the linear version to separate nonlinear accessibility from probe memorization.


In [ ]:
mlp_legal_states, mlp_legal_results = train_legal_mask_probes(
    new_probe_train_data,
    new_probe_val_data,
    layers=PROBE_LAYERS,
    device=device,
    epochs=PROBE_EPOCHS,
    batch_size=PROBE_BATCH_SIZE,
    lr=PROBE_LR,
    weight_decay=0.0,
    probe_kind='mlp',
)

mlp_legal_path = _unique_artifact_path(JEPA_OUT_DIR / f'mlp_legal_mask_probe__{DRIVE_CKPT.stem}.pt')
mlp_legal_json = _unique_artifact_path(JEPA_OUT_DIR / f'eval_legal_mask_mlp__{DRIVE_CKPT.stem}.json')
torch.save(mlp_legal_states, mlp_legal_path)
mlp_legal_json.write_text(json.dumps(jsonable(mlp_legal_results), indent=2), encoding='utf-8')
print('saved:', mlp_legal_path)
print('saved:', mlp_legal_json)


## G. Combined evaluation summary

This summary compares the existing linear evaluations with the new nonlinear and legal-mask probes. The next-move table remains stochastic-bounded; the legal-mask table is the direct legality readout.


In [ ]:

def _latest_json_or_none(pattern: str):
    matches = sorted(JEPA_OUT_DIR.glob(pattern))
    if not matches and 'HEAD_OUT_DIR' in globals():
        matches = sorted(HEAD_OUT_DIR.glob(pattern))
    return matches[-1] if matches else None


def _load_json_artifact(path: Path, label: str):
    if path is None or not Path(path).exists():
        raise NameError(f'{label} is missing in this runtime and no saved JSON artifact was found. Run the corresponding evaluation cell first.')
    print(f'Loaded {label} from:', path)
    return json.loads(Path(path).read_text(encoding='utf-8'))

# Make the summary runnable after a runtime restart or when earlier cells were skipped.
if 'results_next_move' not in globals():
    results_next_move = _load_json_artifact(HEAD_OUT_DIR / f'eval_next_move__{DRIVE_CKPT.stem}.json', 'results_next_move')
if 'mlp_next_move_results' not in globals():
    mlp_next_move_results = _load_json_artifact(_latest_json_or_none(f'eval_next_move_mlp__{DRIVE_CKPT.stem}*.json'), 'mlp_next_move_results')
if 'probe_results' not in globals():
    probe_results = _load_json_artifact(JEPA_OUT_DIR / f'eval_board_state_probes__{DRIVE_CKPT.stem}.json', 'probe_results')
if 'mlp_board_results' not in globals():
    mlp_board_results = _load_json_artifact(_latest_json_or_none(f'eval_board_mlp__{DRIVE_CKPT.stem}*.json'), 'mlp_board_results')
if 'linear_legal_results' not in globals():
    linear_legal_results = _load_json_artifact(_latest_json_or_none(f'eval_legal_mask_linear__{DRIVE_CKPT.stem}*.json'), 'linear_legal_results')
if 'mlp_legal_results' not in globals():
    mlp_legal_results = _load_json_artifact(_latest_json_or_none(f'eval_legal_mask_mlp__{DRIVE_CKPT.stem}*.json'), 'mlp_legal_results')
if 'PROBE_LAYERS' not in globals():
    PROBE_LAYERS = tuple(int(layer) for layer in probe_results['layers'])
if 'PLOT_DIR' not in globals():
    PLOT_DIR = JEPA_OUT_DIR / 'eval_plots'
    PLOT_DIR.mkdir(parents=True, exist_ok=True)

def _pct_value(x):
    if x is None:
        return 'n/a'
    try:
        xf = float(x)
    except Exception:
        return 'n/a'
    if math.isnan(xf):
        return 'nan'
    return f'{100 * xf:5.1f}%'


def _fmt_float(x, digits=3):
    if x is None:
        return 'n/a'
    try:
        xf = float(x)
    except Exception:
        return 'n/a'
    if math.isnan(xf):
        return 'nan'
    return f'{xf:.{digits}f}'


def _best_board_layer(results, split='val'):
    best_layer = None
    best_metric = -1.0
    for layer in results['layers']:
        m = results['metrics'][str(layer)]
        if split in m:
            rel_acc = m[split]['relative']['accuracy']
        else:
            rel_acc = m['relative']['accuracy']
        if rel_acc > best_metric:
            best_metric = rel_acc
            best_layer = int(layer)
    if best_layer is None:
        return None, None
    m = results['metrics'][str(best_layer)]
    return best_layer, (m[split] if split in m else m)


def _best_legal_layer(results, split='val'):
    best_layer = None
    best_metric = -1.0
    for layer in results['layers']:
        m = results['metrics'][str(layer)][split]
        score = float(m['top1_legal_hit_rate'])
        if score > best_metric:
            best_metric = score
            best_layer = int(layer)
    return best_layer, results['metrics'][str(best_layer)][split]

linear_next = results_next_move
mlp_next = mlp_next_move_results['final_val']
linear_board_layer, linear_board_best = _best_board_layer(probe_results, split='val')
mlp_board_layer, mlp_board_best = _best_board_layer(mlp_board_results, split='val')
linear_legal_layer, linear_legal_best = _best_legal_layer(linear_legal_results, split='val')
mlp_legal_layer, mlp_legal_best = _best_legal_layer(mlp_legal_results, split='val')

print('=' * 64)
print(f'JEPA evaluation summary for checkpoint: {DRIVE_CKPT.name}')
print('=' * 64)
print('\nNEXT-MOVE PREDICTION (stochastic-bounded, ceiling ~16%)')
print(f"{'':18s} {'loss':>8s} {'top-1':>8s} {'top-3':>8s} {'top-5':>8s}")
print(f"{'Linear head':18s} {linear_next['loss']:8.3f} {_pct_value(linear_next['top1_accuracy']):>8s} {_pct_value(linear_next['top3_accuracy']):>8s} {_pct_value(linear_next['top5_accuracy']):>8s}")
print(f"{'MLP head':18s} {mlp_next['loss']:8.3f} {_pct_value(mlp_next['top1_accuracy']):>8s} {_pct_value(mlp_next['top3_accuracy']):>8s} {_pct_value(mlp_next['top5_accuracy']):>8s}")
print(f"{'Delta (MLP-Lin)':18s} {'----':>8s} {_pct_value(mlp_next['top1_accuracy'] - linear_next['top1_accuracy']):>8s} {_pct_value(mlp_next['top3_accuracy'] - linear_next['top3_accuracy']):>8s} {_pct_value(mlp_next['top5_accuracy'] - linear_next['top5_accuracy']):>8s}")

print('\nBOARD REPRESENTATION (per-layer best relative accuracy)')
print(
    f"Linear probe:  L{linear_board_layer}  "
    f"rel={_pct_value(linear_board_best['relative']['accuracy'])} "
    f"abs={_pct_value(linear_board_best['absolute']['accuracy'])} "
    f"rel_occ={_pct_value(linear_board_best['relative']['occupied_accuracy'])}"
)
print(
    f"MLP probe:     L{mlp_board_layer}  "
    f"rel={_pct_value(mlp_board_best['relative']['accuracy'])} "
    f"abs={_pct_value(mlp_board_best['absolute']['accuracy'])} "
    f"rel_occ={_pct_value(mlp_board_best['relative']['occupied_accuracy'])}"
)

print('\nLEGAL-MASK PROBE (direct legality readout)')
print(f"{'':18s} {'F1':>8s} {'top-1-legal':>14s} {'top-3-legal':>14s} {'AUROC':>8s}")
print(f"{'Linear (best L'+str(linear_legal_layer)+')':18s} {_fmt_float(linear_legal_best['f1']):>8s} {_pct_value(linear_legal_best['top1_legal_hit_rate']):>14s} {_pct_value(linear_legal_best['top3_legal_hit_rate']):>14s} {_fmt_float(linear_legal_best['auroc']):>8s}")
print(f"{'MLP (best L'+str(mlp_legal_layer)+')':18s} {_fmt_float(mlp_legal_best['f1']):>8s} {_pct_value(mlp_legal_best['top1_legal_hit_rate']):>14s} {_pct_value(mlp_legal_best['top3_legal_hit_rate']):>14s} {_fmt_float(mlp_legal_best['auroc']):>8s}")
print('=' * 64)

print('\nPer-layer board representation table')
print(f"{'layer':>5s} {'lin_abs':>9s} {'lin_rel':>9s} {'mlp_abs':>9s} {'mlp_rel':>9s} {'mlp_train_rel':>13s}")
for layer in PROBE_LAYERS:
    lin = probe_results['metrics'][str(layer)]
    mlp = mlp_board_results['metrics'][str(layer)]
    print(
        f"{layer:5d} "
        f"{_pct_value(lin['absolute']['accuracy']):>9s} {_pct_value(lin['relative']['accuracy']):>9s} "
        f"{_pct_value(mlp['val']['absolute']['accuracy']):>9s} {_pct_value(mlp['val']['relative']['accuracy']):>9s} "
        f"{_pct_value(mlp['train']['relative']['accuracy']):>13s}"
    )

print('\nPer-layer linear legal-mask table')
print(f"{'layer':>5s} {'train_top1':>11s} {'val_f1':>8s} {'val_top1':>10s} {'val_top3':>10s} {'val_auroc':>10s}")
for layer in PROBE_LAYERS:
    m = linear_legal_results['metrics'][str(layer)]
    print(
        f"{layer:5d} {_pct_value(m['train']['top1_legal_hit_rate']):>11s} "
        f"{_fmt_float(m['val']['f1']):>8s} {_pct_value(m['val']['top1_legal_hit_rate']):>10s} "
        f"{_pct_value(m['val']['top3_legal_hit_rate']):>10s} {_fmt_float(m['val']['auroc']):>10s}"
    )

print('\nPer-layer MLP legal-mask table')
print(f"{'layer':>5s} {'train_top1':>11s} {'val_f1':>8s} {'val_top1':>10s} {'val_top3':>10s} {'val_auroc':>10s}")
for layer in PROBE_LAYERS:
    m = mlp_legal_results['metrics'][str(layer)]
    print(
        f"{layer:5d} {_pct_value(m['train']['top1_legal_hit_rate']):>11s} "
        f"{_fmt_float(m['val']['f1']):>8s} {_pct_value(m['val']['top1_legal_hit_rate']):>10s} "
        f"{_pct_value(m['val']['top3_legal_hit_rate']):>10s} {_fmt_float(m['val']['auroc']):>10s}"
    )

# Optional plots for quick visual comparison.
layers = list(PROBE_LAYERS)
plt.figure(figsize=(10, 4.5))
plt.subplot(1, 2, 1)
plt.plot(layers, [probe_results['metrics'][str(l)]['relative']['accuracy'] for l in layers], marker='o', label='linear relative')
plt.plot(layers, [mlp_board_results['metrics'][str(l)]['val']['relative']['accuracy'] for l in layers], marker='o', label='MLP relative')
plt.xlabel('Layer')
plt.ylabel('Board relative accuracy')
plt.ylim(0.0, 1.02)
plt.grid(True, alpha=0.3)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(layers, [linear_legal_results['metrics'][str(l)]['val']['top1_legal_hit_rate'] for l in layers], marker='o', label='linear legal top-1')
plt.plot(layers, [mlp_legal_results['metrics'][str(l)]['val']['top1_legal_hit_rate'] for l in layers], marker='o', label='MLP legal top-1')
plt.xlabel('Layer')
plt.ylabel('Legal-mask top-1 hit')
plt.ylim(0.0, 1.02)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
summary_plot_path = PLOT_DIR / f'jepa_probe_summary_mlp_legal__{DRIVE_CKPT.stem}.png'
plt.savefig(summary_plot_path, dpi=160)
plt.show()
print('saved:', summary_plot_path)


## H. Predictor-Space Evaluation Rationale

For JEPA models, the predictor is trained to map the context representation toward a future latent target. The raw context embedding and the predictor-transformed embedding can therefore expose different information to downstream readouts.

These cells keep the entire JEPA model frozen and compare downstream heads/probes on two representation sources:

1. `context`: the final context encoder summary.
2. `predictor`: the frozen predictor output applied to that context summary.

For multi-position predictors, the first predicted future position is used. This keeps the diagnostic simple and matches the next-step legal-move readout.


## I. Predictor Feature Extraction Helpers


In [ ]:
# Predictor-space helpers. These do not modify or fine-tune JEPA.
import gc
import math
import shutil
import json

import torch
import random
from collections import defaultdict
from contextlib import nullcontext
from pathlib import Path

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm


# Make this section runnable after Configure + Pick Checkpoint, even if the explicit restore cell was skipped.
if 'jepa_model' not in globals():
    if 'DRIVE_CKPT' not in globals() or 'LOCAL_HEAD_DIR' not in globals():
        raise NameError('jepa_model is missing. Run sections 2 and 3 first so DRIVE_CKPT/LOCAL_HEAD_DIR are defined.')
    import othello_research.training.train_jepa as tjm
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') if 'device' not in globals() else device
    LOCAL_CKPT = LOCAL_HEAD_DIR / f'jepa_{DRIVE_CKPT.stem}.pt'
    if not LOCAL_CKPT.exists():
        print(f'Restoring {DRIVE_CKPT.stat().st_size / 1e6:.1f} MB from Drive...')
        shutil.copy2(DRIVE_CKPT, LOCAL_CKPT)
        print('done')
    ckpt = torch.load(LOCAL_CKPT, map_location=device)
    jepa_model, jepa_cfg = tjm.build_model_from_checkpoint(ckpt)
    jepa_model.load_state_dict(ckpt['model'])
    jepa_model.to(device)
    jepa_model.eval()
    print('Loaded JEPA model for predictor-space evaluation:', DRIVE_CKPT.name)

if 'MLPProbe' not in globals():
    class MLPProbe(nn.Module):
        def __init__(self, d_model, out_dim, hidden_dim=512, n_layers=1, dropout=0.1):
            super().__init__()
            layers = [nn.Linear(d_model, hidden_dim), nn.GELU(), nn.Dropout(dropout)]
            for _ in range(n_layers - 1):
                layers += [nn.Linear(hidden_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout)]
            layers += [nn.Linear(hidden_dim, out_dim)]
            self.net = nn.Sequential(*layers)

        def forward(self, x):
            return self.net(x)

if 'MLP_PROBE_HIDDEN_DIM' not in globals():
    MLP_PROBE_HIDDEN_DIM = 512
if 'MLP_PROBE_N_LAYERS' not in globals():
    MLP_PROBE_N_LAYERS = 1
if 'MLP_PROBE_DROPOUT' not in globals():
    MLP_PROBE_DROPOUT = 0.1

if '_unique_artifact_path' not in globals():
    def _unique_artifact_path(path: Path) -> Path:
        path = Path(path)
        if not path.exists():
            return path
        for idx in range(2, 1000):
            candidate = path.with_name(f'{path.stem}__rerun{idx}{path.suffix}')
            if not candidate.exists():
                return candidate
        raise RuntimeError(f'Could not find a free artifact filename for {path}')

def _raw_jepa_model(jepa_model):
    return jepa_model._orig_mod if hasattr(jepa_model, '_orig_mod') else jepa_model

def _amp_predictor_eval_context(device: torch.device, precision: str = 'bf16'):
    if device.type == 'cuda' and precision in {'bf16', 'fp16'}:
        dtype = torch.bfloat16 if precision == 'bf16' else torch.float16
        return torch.autocast(device_type='cuda', dtype=dtype, enabled=True)
    return nullcontext()

def _call_frozen_predictor(raw_model, features: torch.Tensor, target_positions: torch.Tensor | None = None) -> torch.Tensor:
    try:
        return raw_model.predictor(features)
    except TypeError as exc:
        if target_positions is None:
            raise TypeError(
                'This predictor requires target_positions. '
                'Provide absolute future positions for predictor-space extraction.'
            ) from exc
        return raw_model.predictor(features, target_positions)

def _normalize_predictor_output(p: torch.Tensor) -> torch.Tensor:
    if p.ndim == 2:
        return p
    if p.ndim == 3:
        if p.shape[1] == 1:
            return p.squeeze(1)
        # Multi-position predictor: use first predicted future position.
        return p[:, 0, :]
    raise ValueError(f'Unexpected predictor output shape: {tuple(p.shape)}')

@torch.no_grad()
def extract_jepa_features(
    jepa_model,
    x_context,
    representation_source='context',
):
    raw_model = _raw_jepa_model(jepa_model)
    c_states = raw_model.encode_hidden(
        raw_model.context_encoder,
        x_context,
    )
    c_summary = c_states[:, -1, :]

    if representation_source == 'context':
        return c_summary

    if representation_source == 'predictor':
        block_size = int(getattr(raw_model.config, 'block_size', x_context.shape[1]))
        next_pos = min(int(x_context.shape[1]), block_size - 1)
        target_positions = torch.full(
            (x_context.shape[0], 1),
            next_pos,
            dtype=torch.long,
            device=x_context.device,
        )
        p = _call_frozen_predictor(raw_model, c_summary, target_positions)
        return _normalize_predictor_output(p)

    raise ValueError(f'Unknown representation_source: {representation_source}')

@torch.no_grad()
def extract_jepa_sequence_features(
    jepa_model,
    x_context,
    representation_source='context',
):
    """Return one feature vector per input position, shape (B, T, d)."""
    raw_model = _raw_jepa_model(jepa_model)
    c_states = raw_model.encode_hidden(raw_model.context_encoder, x_context)
    if representation_source == 'context':
        return c_states
    if representation_source != 'predictor':
        raise ValueError(f'Unknown representation_source: {representation_source}')

    B, T, d = c_states.shape
    flat = c_states.reshape(B * T, d)
    block_size = int(getattr(raw_model.config, 'block_size', T))
    # Row t predicts the next move at absolute position t+1.
    # The final clamped row is ignored by next-token losses when the target is padding.
    pos = torch.arange(T, device=x_context.device).add(1).clamp(max=block_size - 1)
    target_positions = pos.unsqueeze(0).expand(B, T).reshape(B * T, 1)
    p = _call_frozen_predictor(raw_model, flat, target_positions)
    p = _normalize_predictor_output(p)
    if p.ndim != 2:
        raise ValueError(f'Predictor features must be 2-D after normalization, got {tuple(p.shape)}')
    return p.reshape(B, T, p.shape[-1])

jepa_model.eval()
for parameter in jepa_model.parameters():
    parameter.requires_grad_(False)

print('Predictor-space helpers ready. JEPA is frozen.')


## J. Context MLP Legal-Move Evaluation

The earlier MLP next-move head cell reports exact next-token metrics. This cell adds the legal-move metrics for the same context+MLP head so the final comparison uses legality as the primary next-move diagnostic.


In [ ]:
from othello_research.evaluation.legal_moves import evaluate_legal_moves

if 'train_chunks' not in globals() or 'val_chunks' not in globals():
    all_chunks = sorted(DATA_DIR.glob('*.pickle'))
    assert len(all_chunks) > N_TRAIN_CHUNKS, f'Need validation chunks in {DATA_DIR}'
    train_chunks = all_chunks[:N_TRAIN_CHUNKS]
    val_chunks = all_chunks[N_TRAIN_CHUNKS:]

if 'n_games' not in globals():
    n_games = 5000
if 'k_values' not in globals():
    k_values = (1, 2, 3, 5)
if 'batch_size' not in globals():
    batch_size = 64

context_mlp_legal_json_base = HEAD_OUT_DIR / f'eval_legal_moves_mlp__context__{DRIVE_CKPT.stem}.json'
if 'context_mlp_legal_results' not in globals():
    if context_mlp_legal_json_base.exists():
        context_mlp_legal_results = json.loads(context_mlp_legal_json_base.read_text(encoding='utf-8'))
        print('Loaded context+MLP legal metrics:', context_mlp_legal_json_base)
    elif 'mlp_next_move_model' in globals():
        context_mlp_legal_results = evaluate_legal_moves(
            model=mlp_next_move_model,
            val_chunks=val_chunks,
            board_size=BOARD_SIZE,
            n_games=n_games,
            device=device,
            k_values=k_values,
            batch_size=batch_size,
        )
        context_mlp_legal_json = _unique_artifact_path(context_mlp_legal_json_base)
        payload = {
            'checkpoint_path': str(DRIVE_CKPT),
            'representation_source': 'context',
            'head_type': 'mlp_next_move',
            'hidden_dim': MLP_PROBE_HIDDEN_DIM,
            'n_layers': MLP_PROBE_N_LAYERS,
            'dropout': MLP_PROBE_DROPOUT,
            'val_chunks': [str(p) for p in val_chunks],
            'legal_metrics': context_mlp_legal_results,
        }
        context_mlp_legal_json.write_text(json.dumps(jsonable(payload), indent=2), encoding='utf-8')
        print('saved:', context_mlp_legal_json)
    else:
        print('Context MLP head is not in memory and no saved legal JSON exists. Run Section B first if you need context+MLP legal metrics.')

if 'context_mlp_legal_results' in globals():
    # Saved artifacts wrap the metrics with metadata. Normalize both fresh
    # in-memory results and loaded JSON payloads to the same flat structure.
    context_mlp_legal_results = context_mlp_legal_results.get(
        'legal_metrics',
        context_mlp_legal_results,
    )
    print('Context + MLP legal top-1:', context_mlp_legal_results['top1_legal_per_token'])
    print('Context + MLP legal top-3:', context_mlp_legal_results['topk_legal'].get(3, context_mlp_legal_results['topk_legal'].get('3')))
    print('Context + MLP legal top-5:', context_mlp_legal_results['topk_legal'].get(5, context_mlp_legal_results['topk_legal'].get('5')))
    print('Context + MLP legal prob mass:', context_mlp_legal_results['legal_prob_mass_per_token'])


## K. Predictor-Based Next-Move Heads

These heads train on frozen predictor-space sequence features. The loss is still exact next-token cross-entropy, but the primary reported validation metrics are legal-move metrics.


In [ ]:
from othello_research.datasets.dataset import TARGET_PAD
from othello_research.evaluation.linear_head import (
    LinearHeadTrainConfig,
    _make_loader,
    evaluate_next_move_head,
)

class JEPAPredictorNextMoveHead(nn.Module):
    """Frozen JEPA context+predictor features plus trainable next-token head."""

    def __init__(
        self,
        jepa_model,
        *,
        head_type: str,
        hidden_dim: int = 512,
        n_layers: int = 1,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.jepa_model = jepa_model
        self.raw_model = _raw_jepa_model(jepa_model)
        self.encoder_config = self.raw_model.config
        for parameter in self.raw_model.parameters():
            parameter.requires_grad_(False)
        self.raw_model.eval()

        d_model = int(self.encoder_config.d_model)
        vocab_size = int(self.encoder_config.vocab_size)
        if head_type == 'linear':
            self.head = nn.Linear(d_model, vocab_size)
        elif head_type == 'mlp':
            self.head = MLPProbe(d_model, vocab_size, hidden_dim=hidden_dim, n_layers=n_layers, dropout=dropout)
        else:
            raise ValueError(f'Unknown head_type={head_type}')
        self.head_type = head_type

    @property
    def config(self):
        return self.encoder_config

    def train(self, mode: bool = True):
        super().train(mode)
        self.raw_model.eval()
        return self

    def forward(self, idx: torch.Tensor, targets: torch.Tensor | None = None):
        with torch.no_grad():
            features = extract_jepa_sequence_features(
                self.raw_model,
                idx,
                representation_source='predictor',
            )
        logits = self.head(features)
        if targets is None:
            return logits, None
        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            targets.reshape(-1),
            ignore_index=TARGET_PAD,
        )
        return logits, loss


def _exact_next_move_batch_metrics(logits: torch.Tensor, targets: torch.Tensor, loss: torch.Tensor):
    mask = targets != TARGET_PAD
    n_tokens = int(mask.sum().item())
    if n_tokens == 0:
        return {'loss_sum': 0.0, 'tokens': 0, 'top1': 0, 'top3': 0, 'top5': 0}
    valid_logits = logits[mask]
    valid_targets = targets[mask]
    preds = valid_logits.argmax(dim=-1)
    k3 = min(3, logits.size(-1))
    k5 = min(5, logits.size(-1))
    top3 = torch.topk(valid_logits, k=k3, dim=-1).indices
    top5 = torch.topk(valid_logits, k=k5, dim=-1).indices
    return {
        'loss_sum': float(loss.detach().cpu().item()) * n_tokens,
        'tokens': n_tokens,
        'top1': int((preds == valid_targets).sum().item()),
        'top3': int((top3 == valid_targets.unsqueeze(-1)).any(dim=-1).sum().item()),
        'top5': int((top5 == valid_targets.unsqueeze(-1)).any(dim=-1).sum().item()),
    }


def _exact_totals_to_metrics(totals):
    n = max(1, int(totals.get('tokens', 0)))
    return {
        'loss': float(totals.get('loss_sum', 0.0) / n),
        'top1_accuracy': float(totals.get('top1', 0) / n),
        'top3_accuracy': float(totals.get('top3', 0) / n),
        'top5_accuracy': float(totals.get('top5', 0) / n),
        'accuracy': float(totals.get('top1', 0) / n),
        'n_tokens': int(totals.get('tokens', 0)),
    }


def train_predictor_next_move_head(
    jepa_model,
    train_chunks,
    val_chunks,
    *,
    head_type: str,
    board_size: int,
    device: torch.device,
    batch_size: int = 512,
    lr: float = 1e-3,
    weight_decay: float = 0.0,
    epochs: int = 1,
    eval_chunks: int = 1,
    legal_eval_games: int = 5000,
    legal_eval_batch_size: int = 64,
    precision: str = 'bf16',
):
    model = JEPAPredictorNextMoveHead(
        jepa_model,
        head_type=head_type,
        hidden_dim=MLP_PROBE_HIDDEN_DIM,
        n_layers=MLP_PROBE_N_LAYERS,
        dropout=MLP_PROBE_DROPOUT,
    ).to(device)
    optimizer = torch.optim.AdamW(model.head.parameters(), lr=lr, weight_decay=weight_decay)
    cfg = LinearHeadTrainConfig(
        train_chunks=train_chunks,
        out_dir=str(LOCAL_HEAD_DIR),
        board_size=board_size,
        batch_size=batch_size,
        learning_rate=lr,
        weight_decay=weight_decay,
        n_chunks=0,
        precision=precision,
    )
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        totals = defaultdict(float)
        pbar = tqdm(train_chunks, desc=f'predictor {head_type} next-move epoch {epoch}/{epochs}', unit='chunk')
        for chunk_path in pbar:
            loader, games, dataset = _make_loader(chunk_path, model, cfg, shuffle=True)
            for x_cpu, y_cpu in tqdm(loader, desc=f'{head_type} {Path(chunk_path).name}', leave=False):
                x = x_cpu.to(device, non_blocking=device.type == 'cuda')
                y = y_cpu.to(device, non_blocking=device.type == 'cuda')
                optimizer.zero_grad(set_to_none=True)
                with _amp_predictor_eval_context(device, precision):
                    logits, loss = model(x, y)
                if loss is None:
                    raise RuntimeError('Predictor next-move training expected a loss')
                loss.backward()
                optimizer.step()
                for key, value in _exact_next_move_batch_metrics(logits, y, loss).items():
                    totals[key] += value
            del loader, dataset, games
            gc.collect()
            if device.type == 'cuda':
                torch.cuda.empty_cache()
            running = _exact_totals_to_metrics(totals)
            pbar.set_postfix(loss=f"{running['loss']:.4f}", exact_top1=f"{running['top1_accuracy']:.2%}")

        exact_train = _exact_totals_to_metrics(totals)
        exact_val = evaluate_next_move_head(
            model=model,
            chunks=val_chunks,
            board_size=board_size,
            device=device,
            n_chunks=eval_chunks,
            batch_size=batch_size,
            precision=precision,
        )
        history.append({'epoch': epoch, 'exact_train': exact_train, 'exact_val': exact_val})
        print(
            f"predictor+{head_type} epoch={epoch} "
            f"exact_train_top1={exact_train['top1_accuracy']:.2%} "
            f"exact_val_top1={exact_val['top1_accuracy']:.2%}"
        )

    legal_val = evaluate_legal_moves(
        model=model,
        val_chunks=val_chunks,
        board_size=board_size,
        n_games=legal_eval_games,
        device=device,
        k_values=(1, 2, 3, 5),
        batch_size=legal_eval_batch_size,
    )
    return model, history, legal_val

RETRAIN_PREDICTOR_NEXT_MOVE_HEADS = True
PREDICTOR_NEXT_MOVE_EPOCHS = 1
PREDICTOR_NEXT_MOVE_BATCH_SIZE = 512
PREDICTOR_NEXT_MOVE_LR = 1e-3
PREDICTOR_NEXT_MOVE_WEIGHT_DECAY = 0.0
PREDICTOR_NEXT_MOVE_PRECISION = 'bf16'
PREDICTOR_NEXT_MOVE_LEGAL_GAMES = n_games if 'n_games' in globals() else 5000
PREDICTOR_NEXT_MOVE_LEGAL_BATCH = batch_size if 'batch_size' in globals() else 64
PREDICTOR_NEXT_MOVE_EVAL_CHUNKS = NEXT_MOVE_EVAL_CHUNKS if 'NEXT_MOVE_EVAL_CHUNKS' in globals() else 1

predictor_next_move_results = {}
predictor_next_move_models = {}
for head_type in ('linear', 'mlp'):
    head_base = HEAD_OUT_DIR / f'{head_type}_next_move_head__predictor__{DRIVE_CKPT.stem}.pt'
    json_base = HEAD_OUT_DIR / f'eval_next_move_{head_type}__predictor__{DRIVE_CKPT.stem}.json'
    head_path = _unique_artifact_path(head_base) if RETRAIN_PREDICTOR_NEXT_MOVE_HEADS else head_base
    json_path = _unique_artifact_path(json_base) if RETRAIN_PREDICTOR_NEXT_MOVE_HEADS else json_base

    if head_base.exists() and json_base.exists() and not RETRAIN_PREDICTOR_NEXT_MOVE_HEADS:
        model = JEPAPredictorNextMoveHead(
            jepa_model,
            head_type=head_type,
            hidden_dim=MLP_PROBE_HIDDEN_DIM,
            n_layers=MLP_PROBE_N_LAYERS,
            dropout=MLP_PROBE_DROPOUT,
        ).to(device)
        payload = torch.load(head_base, map_location=device)
        model.head.load_state_dict(payload['head'])
        result = json.loads(json_base.read_text(encoding='utf-8'))
        print(f'Loaded predictor+{head_type} next-move artifacts:', head_base, json_base)
    else:
        model, history, legal_val = train_predictor_next_move_head(
            jepa_model,
            train_chunks,
            val_chunks,
            head_type=head_type,
            board_size=BOARD_SIZE,
            device=device,
            batch_size=PREDICTOR_NEXT_MOVE_BATCH_SIZE,
            lr=PREDICTOR_NEXT_MOVE_LR,
            weight_decay=PREDICTOR_NEXT_MOVE_WEIGHT_DECAY,
            epochs=PREDICTOR_NEXT_MOVE_EPOCHS,
            eval_chunks=PREDICTOR_NEXT_MOVE_EVAL_CHUNKS,
            legal_eval_games=PREDICTOR_NEXT_MOVE_LEGAL_GAMES,
            legal_eval_batch_size=PREDICTOR_NEXT_MOVE_LEGAL_BATCH,
            precision=PREDICTOR_NEXT_MOVE_PRECISION,
        )
        exact_val = history[-1]['exact_val'] if history else None
        result = {
            'checkpoint_path': str(DRIVE_CKPT),
            'representation_source': 'predictor',
            'head_type': f'{head_type}_next_move',
            'hidden_dim': MLP_PROBE_HIDDEN_DIM if head_type == 'mlp' else None,
            'n_layers': MLP_PROBE_N_LAYERS if head_type == 'mlp' else None,
            'dropout': MLP_PROBE_DROPOUT if head_type == 'mlp' else None,
            'train_chunks': [str(p) for p in train_chunks],
            'val_chunks': [str(p) for p in val_chunks],
            'history': history,
            'exact_next_token_metrics': exact_val,
            'legal_metrics': legal_val,
        }
        torch.save({'head': model.head.state_dict(), 'metadata': result}, head_path)
        json_path.write_text(json.dumps(jsonable(result), indent=2), encoding='utf-8')
        print('saved:', head_path)
        print('saved:', json_path)

    predictor_next_move_models[head_type] = model
    predictor_next_move_results[head_type] = result
    legal = result['legal_metrics']
    print(
        f"Predictor + {head_type} legal: "
        f"top1={legal['top1_legal_per_token']:.2%} "
        f"top3={legal['topk_legal'].get(3, legal['topk_legal'].get('3')):.2%} "
        f"top5={legal['topk_legal'].get(5, legal['topk_legal'].get('5')):.2%} "
        f"mass={legal['legal_prob_mass_per_token']:.2%}"
    )


## L. Predictor-Based Board-State Probes

These probes use a single predictor-space representation: the first future prediction from the frozen predictor applied to the final context summary. The board targets and train/validation chunks match the existing board-state probe convention.


In [ ]:
from othello_research.datasets.move_mapping import build_mappings
from othello_research.probes.board_state import (
    EMPTY as BOARD_EMPTY_LABEL,
    board_state_labels_after_moves,
    load_games_from_chunks,
)

if 'PROBE_TRAIN_GAMES' not in globals():
    PROBE_TRAIN_GAMES = 5000
if 'PROBE_VAL_GAMES' not in globals():
    PROBE_VAL_GAMES = 512
if 'PROBE_BATCH_SIZE' not in globals():
    PROBE_BATCH_SIZE = 64
if 'PROBE_EPOCHS' not in globals():
    PROBE_EPOCHS = 3
if 'PROBE_LR' not in globals():
    PROBE_LR = 1e-3

def _sample_predictor_board_examples(chunks, *, n_games: int, board_size: int, block_size: int, seed: int):
    raw_to_token, _ = build_mappings(board_size)
    games = load_games_from_chunks(chunks, n_games)
    rng = random.Random(seed)
    examples = []
    skipped = 0
    for game_idx, game_raw in enumerate(games):
        raw = list(game_raw[: block_size + 1])
        max_moves = min(len(raw), block_size + 1)
        if max_moves < 2:
            skipped += 1
            continue
        ply = rng.randrange(0, max_moves - 1)
        prefix_raw = raw[: ply + 1]
        tokens = [raw_to_token[int(move)] for move in prefix_raw]
        if any(token < 0 for token in tokens):
            skipped += 1
            continue
        labels_abs, labels_rel = board_state_labels_after_moves(prefix_raw, board_size)
        examples.append({
            'tokens': torch.tensor(tokens, dtype=torch.long),
            'position': len(tokens) - 1,
            'absolute': torch.tensor(labels_abs[-1], dtype=torch.long),
            'relative': torch.tensor(labels_rel[-1], dtype=torch.long),
            'game_idx': game_idx,
            'ply': ply,
        })
    return examples, {'loaded_games': len(games), 'examples': len(examples), 'skipped': skipped}

def collect_predictor_board_dataset(
    jepa_model,
    chunks,
    *,
    n_games: int,
    board_size: int,
    batch_size: int,
    device: torch.device,
    seed: int,
    precision: str = 'bf16',
):
    raw_model = _raw_jepa_model(jepa_model)
    block_size = int(getattr(raw_model.config, 'block_size', board_size ** 2 - 5))
    pad_token = board_size ** 2 - 4
    examples, sample_info = _sample_predictor_board_examples(
        chunks,
        n_games=n_games,
        board_size=board_size,
        block_size=block_size,
        seed=seed,
    )
    if not examples:
        raise ValueError('No predictor board probe examples were sampled')
    features = []
    y_abs = []
    y_rel = []
    metadata = []
    for start in tqdm(range(0, len(examples), batch_size), desc='collect predictor board features'):
        batch = examples[start : start + batch_size]
        max_len = max(len(ex['tokens']) for ex in batch)
        x = torch.full((len(batch), max_len), pad_token, dtype=torch.long)
        positions = torch.empty(len(batch), dtype=torch.long)
        for row, ex in enumerate(batch):
            tokens = ex['tokens']
            x[row, : len(tokens)] = tokens
            positions[row] = int(ex['position'])
            y_abs.append(ex['absolute'])
            y_rel.append(ex['relative'])
            metadata.append({'game_idx': int(ex['game_idx']), 'ply': int(ex['ply'])})
        x = x.to(device)
        positions = positions.to(device)
        with torch.no_grad():
            with _amp_predictor_eval_context(device, precision):
                seq_features = extract_jepa_sequence_features(jepa_model, x, representation_source='predictor')
        row_idx = torch.arange(len(batch), device=device)
        features.append(seq_features[row_idx, positions].detach().float().cpu())
    return {
        'features': torch.cat(features, dim=0),
        'absolute': torch.stack(y_abs, dim=0),
        'relative': torch.stack(y_rel, dim=0),
        'metadata': metadata,
        'sample_info': sample_info,
        'board_size': board_size,
        'n_squares': board_size ** 2,
        'layers': ['predictor'],
    }

class PredictorBoardProbe(nn.Module):
    def __init__(self, d_model: int, n_squares: int, *, probe_type: str):
        super().__init__()
        self.n_squares = n_squares
        if probe_type == 'linear':
            self.proj = nn.Linear(d_model, n_squares * 3)
        elif probe_type == 'mlp':
            self.proj = MLPProbe(
                d_model,
                n_squares * 3,
                hidden_dim=MLP_PROBE_HIDDEN_DIM,
                n_layers=MLP_PROBE_N_LAYERS,
                dropout=MLP_PROBE_DROPOUT,
            )
        else:
            raise ValueError(f'Unknown probe_type={probe_type}')

    def forward(self, x):
        return self.proj(x).view(*x.shape[:-1], self.n_squares, 3)

def _predictor_board_counts(logits, labels):
    preds = logits.argmax(dim=-1)
    valid = torch.ones_like(labels, dtype=torch.bool)
    occupied = valid & (labels != BOARD_EMPTY_LABEL)
    empty = valid & (labels == BOARD_EMPTY_LABEL)
    return {
        'total': int(valid.sum().item()),
        'correct': int(((preds == labels) & valid).sum().item()),
        'occupied_total': int(occupied.sum().item()),
        'occupied_correct': int(((preds == labels) & occupied).sum().item()),
        'empty_total': int(empty.sum().item()),
        'empty_correct': int(((preds == labels) & empty).sum().item()),
    }

def _predictor_counts_to_metrics(counts):
    def ratio(num, den):
        den_value = counts.get(den, 0)
        return float('nan') if den_value == 0 else float(counts.get(num, 0) / den_value)
    return {
        'accuracy': ratio('correct', 'total'),
        'occupied_accuracy': ratio('occupied_correct', 'occupied_total'),
        'empty_accuracy': ratio('empty_correct', 'empty_total'),
        'n_labels': int(counts.get('total', 0)),
        'n_occupied_labels': int(counts.get('occupied_total', 0)),
        'n_empty_labels': int(counts.get('empty_total', 0)),
    }

def _evaluate_predictor_board_pair(abs_probe, rel_probe, X, y_abs, y_rel, *, batch_size, device):
    abs_probe.eval(); rel_probe.eval()
    abs_counts, rel_counts = {}, {}
    total_loss = 0.0
    total_batches = 0
    loader = DataLoader(TensorDataset(X, y_abs, y_rel), batch_size=batch_size, shuffle=False)
    with torch.no_grad():
        for xb, ya, yr in loader:
            xb = xb.to(device)
            ya = ya.to(device)
            yr = yr.to(device)
            abs_logits = abs_probe(xb)
            rel_logits = rel_probe(xb)
            loss = 0.5 * (
                F.cross_entropy(abs_logits.reshape(-1, 3), ya.reshape(-1)) +
                F.cross_entropy(rel_logits.reshape(-1, 3), yr.reshape(-1))
            )
            total_loss += float(loss.detach().cpu())
            total_batches += 1
            for dst, src in ((abs_counts, _predictor_board_counts(abs_logits, ya)), (rel_counts, _predictor_board_counts(rel_logits, yr))):
                for key, value in src.items():
                    dst[key] = dst.get(key, 0) + int(value)
    return {
        'loss': total_loss / max(1, total_batches),
        'absolute': _predictor_counts_to_metrics(abs_counts),
        'relative': _predictor_counts_to_metrics(rel_counts),
    }

def train_predictor_board_probe(train_data, val_data, *, probe_type: str, device, epochs, batch_size, lr, weight_decay):
    X_train = train_data['features']
    X_val = val_data['features']
    y_abs_train = train_data['absolute']
    y_rel_train = train_data['relative']
    y_abs_val = val_data['absolute']
    y_rel_val = val_data['relative']
    d_model = int(X_train.shape[-1])
    n_squares = int(train_data['n_squares'])
    abs_probe = PredictorBoardProbe(d_model, n_squares, probe_type=probe_type).to(device)
    rel_probe = PredictorBoardProbe(d_model, n_squares, probe_type=probe_type).to(device)
    optimizer = torch.optim.AdamW(list(abs_probe.parameters()) + list(rel_probe.parameters()), lr=lr, weight_decay=weight_decay)
    loader = DataLoader(TensorDataset(X_train, y_abs_train, y_rel_train), batch_size=batch_size, shuffle=True)
    history = []
    for epoch in range(1, epochs + 1):
        abs_probe.train(); rel_probe.train()
        epoch_loss = 0.0
        n_batches = 0
        for xb, ya, yr in loader:
            xb = xb.to(device)
            ya = ya.to(device)
            yr = yr.to(device)
            optimizer.zero_grad(set_to_none=True)
            abs_logits = abs_probe(xb)
            rel_logits = rel_probe(xb)
            loss = 0.5 * (
                F.cross_entropy(abs_logits.reshape(-1, 3), ya.reshape(-1)) +
                F.cross_entropy(rel_logits.reshape(-1, 3), yr.reshape(-1))
            )
            loss.backward()
            optimizer.step()
            epoch_loss += float(loss.detach().cpu())
            n_batches += 1
        train_metrics = _evaluate_predictor_board_pair(abs_probe, rel_probe, X_train, y_abs_train, y_rel_train, batch_size=batch_size, device=device)
        val_metrics = _evaluate_predictor_board_pair(abs_probe, rel_probe, X_val, y_abs_val, y_rel_val, batch_size=batch_size, device=device)
        history.append({'epoch': epoch, 'train_loss': epoch_loss / max(1, n_batches), 'train': train_metrics, 'val': val_metrics})
        print(
            f"predictor+{probe_type} board epoch={epoch} "
            f"train_rel={train_metrics['relative']['accuracy']:.2%} "
            f"val_rel={val_metrics['relative']['accuracy']:.2%} "
            f"train_abs={train_metrics['absolute']['accuracy']:.2%} "
            f"val_abs={val_metrics['absolute']['accuracy']:.2%}"
        )
    return {'absolute': abs_probe.cpu().state_dict(), 'relative': rel_probe.cpu().state_dict()}, history

predictor_board_train_data = collect_predictor_board_dataset(
    jepa_model,
    train_chunks,
    n_games=PROBE_TRAIN_GAMES,
    board_size=BOARD_SIZE,
    batch_size=PROBE_BATCH_SIZE,
    device=device,
    seed=NEW_PROBE_SEED if 'NEW_PROBE_SEED' in globals() else 12345,
)
predictor_board_val_data = collect_predictor_board_dataset(
    jepa_model,
    val_chunks,
    n_games=PROBE_VAL_GAMES,
    board_size=BOARD_SIZE,
    batch_size=PROBE_BATCH_SIZE,
    device=device,
    seed=(NEW_PROBE_SEED + 1) if 'NEW_PROBE_SEED' in globals() else 12346,
)

predictor_board_results = {}
for probe_type in ('linear', 'mlp'):
    states, history = train_predictor_board_probe(
        predictor_board_train_data,
        predictor_board_val_data,
        probe_type=probe_type,
        device=device,
        epochs=PROBE_EPOCHS,
        batch_size=PROBE_BATCH_SIZE,
        lr=PROBE_LR,
        weight_decay=0.0,
    )
    result = {
        'checkpoint_path': str(DRIVE_CKPT),
        'representation_source': 'predictor',
        'probe_type': f'{probe_type}_board',
        'hidden_dim': MLP_PROBE_HIDDEN_DIM if probe_type == 'mlp' else None,
        'n_layers': MLP_PROBE_N_LAYERS if probe_type == 'mlp' else None,
        'dropout': MLP_PROBE_DROPOUT if probe_type == 'mlp' else None,
        'layers': ['predictor'],
        'train_chunks': [str(p) for p in train_chunks],
        'val_chunks': [str(p) for p in val_chunks],
        'train_sample_info': predictor_board_train_data['sample_info'],
        'val_sample_info': predictor_board_val_data['sample_info'],
        'metrics': {'predictor': {'train': history[-1]['train'], 'val': history[-1]['val']}},
        'history': {'predictor': history},
    }
    state_path = _unique_artifact_path(JEPA_OUT_DIR / f'{probe_type}_board_probe__predictor__{DRIVE_CKPT.stem}.pt')
    json_path = _unique_artifact_path(JEPA_OUT_DIR / f'eval_board_{probe_type}__predictor__{DRIVE_CKPT.stem}.json')
    torch.save({'states': states, 'metadata': result}, state_path)
    json_path.write_text(json.dumps(jsonable(result), indent=2), encoding='utf-8')
    predictor_board_results[probe_type] = result
    print('saved:', state_path)
    print('saved:', json_path)


## M. Context vs Predictor Summary

This summary makes legal-move metrics the primary next-move diagnostic, then compares board-state probes across context and predictor spaces.


In [ ]:
def _load_json_if_needed(var_name: str, path: Path):
    if var_name in globals():
        return globals()[var_name]
    if not path.exists():
        raise NameError(f'{var_name} is missing and {path} does not exist. Run the corresponding cell first.')
    value = json.loads(path.read_text(encoding='utf-8'))
    globals()[var_name] = value
    print(f'Loaded {var_name} from:', path)
    return value

if 'results_legal' not in globals():
    results_legal = _load_json_if_needed('results_legal', JEPA_OUT_DIR / f'eval_legal_moves__{DRIVE_CKPT.stem}.json')
if 'probe_results' not in globals():
    probe_results = _load_json_if_needed('probe_results', JEPA_OUT_DIR / f'eval_board_state_probes__{DRIVE_CKPT.stem}.json')

if 'context_mlp_legal_results' not in globals():
    context_mlp_path = HEAD_OUT_DIR / f'eval_legal_moves_mlp__context__{DRIVE_CKPT.stem}.json'
    if context_mlp_path.exists():
        context_mlp_payload = json.loads(context_mlp_path.read_text(encoding='utf-8'))
        context_mlp_legal_results = context_mlp_payload.get('legal_metrics', context_mlp_payload)
    else:
        context_mlp_legal_results = None

if 'mlp_board_results' not in globals():
    candidates = sorted(JEPA_OUT_DIR.glob(f'eval_board_mlp__{DRIVE_CKPT.stem}*.json'))
    mlp_board_results = json.loads(candidates[-1].read_text(encoding='utf-8')) if candidates else None

if 'predictor_next_move_results' not in globals():
    predictor_next_move_results = {}
    for head_type in ('linear', 'mlp'):
        candidates = sorted(HEAD_OUT_DIR.glob(f'eval_next_move_{head_type}__predictor__{DRIVE_CKPT.stem}*.json'))
        if candidates:
            predictor_next_move_results[head_type] = json.loads(candidates[-1].read_text(encoding='utf-8'))

if 'predictor_board_results' not in globals():
    predictor_board_results = {}
    for probe_type in ('linear', 'mlp'):
        candidates = sorted(JEPA_OUT_DIR.glob(f'eval_board_{probe_type}__predictor__{DRIVE_CKPT.stem}*.json'))
        if candidates:
            predictor_board_results[probe_type] = json.loads(candidates[-1].read_text(encoding='utf-8'))

def _topk_metric(legal, k):
    if legal is None:
        return None
    values = legal.get('topk_legal', {})
    return values.get(k, values.get(str(k)))

def _pct_or_na(x):
    if x is None:
        return 'n/a'
    try:
        xf = float(x)
    except Exception:
        return 'n/a'
    if math.isnan(xf):
        return 'nan'
    return f'{100 * xf:.2f}%'

def _legal_row(label, legal):
    print(f"{label:<22s} top1={_pct_or_na(None if legal is None else legal.get('top1_legal_per_token')):>8s} "
          f"top3={_pct_or_na(_topk_metric(legal, 3)):>8s} "
          f"top5={_pct_or_na(_topk_metric(legal, 5)):>8s} "
          f"mass={_pct_or_na(None if legal is None else legal.get('legal_prob_mass_per_token')):>8s}")

def _best_context_board(results):
    best_layer = None
    best_acc = -1.0
    best = None
    for layer in results['layers']:
        m = results['metrics'][str(layer)]
        if 'val' in m:
            rel = m['val']['relative']['accuracy']
            payload = m['val']
        else:
            rel = m['relative']['accuracy']
            payload = m
        if rel > best_acc:
            best_acc = rel
            best_layer = layer
            best = payload
    return best_layer, best

def _best_predictor_board(result):
    if result is None:
        return None, None
    m = result['metrics']['predictor']['val']
    return 'predictor', m

print('=' * 70)
print('NEXT-MOVE LEGAL EVALUATION')
print('=' * 70)
_legal_row('Context + linear', results_legal)
_legal_row('Context + MLP', context_mlp_legal_results)
_legal_row('Predictor + linear', predictor_next_move_results.get('linear', {}).get('legal_metrics'))
_legal_row('Predictor + MLP', predictor_next_move_results.get('mlp', {}).get('legal_metrics'))

print('\n' + '=' * 70)
print('BOARD REPRESENTATION PROBES')
print('=' * 70)
ctx_lin_layer, ctx_lin = _best_context_board(probe_results)
ctx_mlp_layer, ctx_mlp = _best_context_board(mlp_board_results) if mlp_board_results else (None, None)
pred_lin_layer, pred_lin = _best_predictor_board(predictor_board_results.get('linear'))
pred_mlp_layer, pred_mlp = _best_predictor_board(predictor_board_results.get('mlp'))

def _board_row(label, layer, metrics):
    if metrics is None:
        print(f'{label:<22s} best layer=n/a acc=n/a rel=n/a abs=n/a')
        return
    print(
        f"{label:<22s} best layer={layer} "
        f"rel={_pct_or_na(metrics['relative']['accuracy'])} "
        f"abs={_pct_or_na(metrics['absolute']['accuracy'])} "
        f"rel_occ={_pct_or_na(metrics['relative']['occupied_accuracy'])}"
    )

_board_row('Context + linear', ctx_lin_layer, ctx_lin)
_board_row('Context + MLP', ctx_mlp_layer, ctx_mlp)
_board_row('Predictor + linear', pred_lin_layer, pred_lin)
_board_row('Predictor + MLP', pred_mlp_layer, pred_mlp)

if 'PLOT_DIR' not in globals():
    PLOT_DIR = JEPA_OUT_DIR / 'eval_plots'
    PLOT_DIR.mkdir(parents=True, exist_ok=True)

plot_layers = [int(layer) for layer in probe_results['layers']]
plt.figure(figsize=(10, 4.5))
plt.subplot(1, 2, 1)
plt.plot(plot_layers, [probe_results['metrics'][str(l)]['relative']['accuracy'] for l in plot_layers], marker='o', label='context linear')
if mlp_board_results:
    plt.plot(plot_layers, [mlp_board_results['metrics'][str(l)]['val']['relative']['accuracy'] for l in plot_layers], marker='o', label='context MLP')
if pred_lin:
    plt.axhline(pred_lin['relative']['accuracy'], linestyle='--', label='predictor linear')
if pred_mlp:
    plt.axhline(pred_mlp['relative']['accuracy'], linestyle='--', label='predictor MLP')
plt.xlabel('Context encoder layer')
plt.ylabel('Relative board accuracy')
plt.ylim(0.0, 1.02)
plt.grid(True, alpha=0.3)
plt.legend()

plt.subplot(1, 2, 2)
labels = ['ctx lin', 'ctx mlp', 'pred lin', 'pred mlp']
values = [
    results_legal.get('top1_legal_per_token'),
    None if context_mlp_legal_results is None else context_mlp_legal_results.get('top1_legal_per_token'),
    predictor_next_move_results.get('linear', {}).get('legal_metrics', {}).get('top1_legal_per_token'),
    predictor_next_move_results.get('mlp', {}).get('legal_metrics', {}).get('top1_legal_per_token'),
]
plt.bar(labels, [0.0 if v is None else float(v) for v in values])
plt.ylabel('Top-1 legal accuracy')
plt.ylim(0.0, 1.02)
plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
predictor_summary_plot = PLOT_DIR / f'context_vs_predictor_summary__{DRIVE_CKPT.stem}.png'
plt.savefig(predictor_summary_plot, dpi=160)
plt.show()
print('saved:', predictor_summary_plot)


## 16. Side-by-side comparison with AR

In [ ]:

# Make this comparison runnable after skipped cells or a runtime restart.
if 'results_legal' not in globals():
    legal_path = JEPA_OUT_DIR / f'eval_legal_moves__{DRIVE_CKPT.stem}.json'
    if not legal_path.exists():
        raise NameError(f'results_legal is missing and {legal_path} does not exist. Run the legal-move evaluation cell first.')
    results_legal = json.loads(legal_path.read_text(encoding='utf-8'))
    print('Loaded results_legal from:', legal_path)
if 'probe_results' not in globals():
    probe_path = JEPA_OUT_DIR / f'eval_board_state_probes__{DRIVE_CKPT.stem}.json'
    if not probe_path.exists():
        raise NameError(f'probe_results is missing and {probe_path} does not exist. Run the linear board-probe cell first.')
    probe_results = json.loads(probe_path.read_text(encoding='utf-8'))
    print('Loaded probe_results from:', probe_path)
if 'PLOT_DIR' not in globals():
    PLOT_DIR = JEPA_OUT_DIR / 'eval_plots'
    PLOT_DIR.mkdir(parents=True, exist_ok=True)

AR_OUT_DIR = RUNS_DIR / 'run_001'

ar_legal = json.loads((AR_OUT_DIR / 'eval_legal_moves__final.json').read_text())
ar_probes = json.loads((AR_OUT_DIR / 'eval_board_state_probes__final.json').read_text())

jepa_legal = results_legal
jepa_probes = probe_results

print('=' * 70)
print('AR vs JEPA-Linear  (8x8, 20M games, same compute)')
print('=' * 70)
print('{:<40s} {:>12s} {:>12s}'.format('Metric', 'AR', 'JEPA'))
print('-' * 70)
print(f"{'Top-1 legal (per-token)':<40s} "
      f"{ar_legal['top1_legal_per_token']:12.4%} "
      f"{jepa_legal['top1_legal_per_token']:12.4%}")
print(f"{'Legal prob mass (per-token)':<40s} "
      f"{ar_legal['legal_prob_mass_per_token']:12.4%} "
      f"{jepa_legal['legal_prob_mass_per_token']:12.4%}")

ar_abs_best = max(ar_probes['metrics'][str(layer)]['absolute']['accuracy'] for layer in ar_probes['layers'])
ar_rel_best = max(ar_probes['metrics'][str(layer)]['relative']['accuracy'] for layer in ar_probes['layers'])
jepa_abs_best = max(jepa_probes['metrics'][str(layer)]['absolute']['accuracy'] for layer in jepa_probes['layers'])
jepa_rel_best = max(jepa_probes['metrics'][str(layer)]['relative']['accuracy'] for layer in jepa_probes['layers'])
print(f"{'Absolute probe (best layer)':<40s} {ar_abs_best:12.4%} {jepa_abs_best:12.4%}")
print(f"{'Relative probe (best layer)':<40s} {ar_rel_best:12.4%} {jepa_rel_best:12.4%}")

ar_layers = ar_probes['layers']
jepa_layers = jepa_probes['layers']
ar_abs = [ar_probes['metrics'][str(layer)]['absolute']['accuracy'] for layer in ar_layers]
ar_rel = [ar_probes['metrics'][str(layer)]['relative']['accuracy'] for layer in ar_layers]
jepa_abs = [jepa_probes['metrics'][str(layer)]['absolute']['accuracy'] for layer in jepa_layers]
jepa_rel = [jepa_probes['metrics'][str(layer)]['relative']['accuracy'] for layer in jepa_layers]

plt.figure(figsize=(8, 4.8))
plt.plot(ar_layers, ar_abs, marker='o', linestyle='-', label='AR absolute')
plt.plot(ar_layers, ar_rel, marker='o', linestyle='-', label='AR relative')
plt.plot(jepa_layers, jepa_abs, marker='o', linestyle='--', label='JEPA absolute')
plt.plot(jepa_layers, jepa_rel, marker='o', linestyle='--', label='JEPA relative')
plt.xlabel('Layer')
plt.ylabel('Board-state probe accuracy')
plt.ylim(0.0, 1.02)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
path = PLOT_DIR / f'ar_vs_jepa_probe_accuracy__{DRIVE_CKPT.stem}.png'
plt.savefig(path, dpi=160)
plt.show()
print('saved:', path)

In [ ]:
from google.colab import runtime
runtime.unassign()